# 머신러닝 파일별 5 · 4-1 로지스틱 회귀 · Sigmoid · 확률 · 임곗값

**한 줄 요약**

- '확률로 분류'한다는 건 무슨 뜻인가?
- 점수 z를 Sigmoid로 확률로 바꾸고, 그 확률로 임곗값 · 순위(AUC)
- 계수 해석까지 하는 로지스틱 회귀를 타이타닉 3분할로 확인

- **원본** · `6.머닝러신/03_전처리_평가지표_Titanic/4-1_로지스틱_회귀.ipynb` (원본 파일은 그대로 둠)
- **종합 정리본** · `00_최소선/05_머신러닝` 머신러닝-3 (평가지표 · 로지스틱 · 임곗값) 3 · 4 · 7장

## 🎯 학습 목표

이 노트북을 끝내면 **자료를 보지 않고** 다음을 할 수 있어야 함

1. σ(z) = 1 / (1 + e⁻ᶻ)로 z = 0 ⇔ 확률 0.5를 설명한다
2. `predict`와 `predict_proba`의 관계 · 교차 엔트로피가 '확신하고 틀리기'를 6.46배 벌주는 것을 말한다
3. 로지스틱이 Dummy와 정확도가 같아도 확률 · AUC를 준다는 점을 설명한다
4. 선언한 규칙(validation F1 최대)으로 임곗값을 고르고, 줄다리기가 '경향'임을 표로 보인다
5. 계수 부호 · 6조합 확률로 근거를 대고, test는 마지막에 한 번만 평가한다

## 어디에 쓰나

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">실제 상황</th>
<th style="text-align: left;">이 권에서 배우는 것</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">모델이 낸 확률을 사람에게 설명할 때</td>
<td style="text-align: left;">Sigmoid · 확률 ≠ 정답 보장 (1 · 4장)</td>
</tr>
<tr>
<td style="text-align: left;">'정확도가 기준선과 같다'는 결과를 받았을 때</td>
<td style="text-align: left;">AUC · 확률이 있는지 (2-4)</td>
</tr>
<tr>
<td style="text-align: left;">놓침 · 오탐의 균형을 정할 때</td>
<td style="text-align: left;">validation에서 임곗값 (3장)</td>
</tr>
<tr>
<td style="text-align: left;">왜 그렇게 예측했는지 물을 때</td>
<td style="text-align: left;">계수 · 조합별 확률 (5장)</td>
</tr>
</tbody>
</table>

## 학습 지도

<img src="images/pfm5_map.png" alt="4-1 로지스틱 학습 지도" width="763">

**예상 학습 시간** · 개념 1시간 30분 + 실습 1시간 (전체 실행 20초 이내 · GPU 필요 없음)

## 목차

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">장</th>
<th style="text-align: left;">제목</th>
<th style="text-align: left;">이 장의 질문</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">0</td>
<td style="text-align: left;">준비</td>
<td style="text-align: left;">타이타닉 891명 · 원본 목표</td>
</tr>
<tr>
<td style="text-align: left;">1</td>
<td style="text-align: left;">Sigmoid</td>
<td style="text-align: left;">모델 점수 z는 어떻게 0 ~ 1의 확률이 되는가?</td>
</tr>
<tr>
<td style="text-align: left;">2</td>
<td style="text-align: left;">로지스틱 회귀 학습</td>
<td style="text-align: left;">로지스틱은 어떻게 학습하고, 규칙 Dummy와 무엇이 다른가?</td>
</tr>
<tr>
<td style="text-align: left;">3</td>
<td style="text-align: left;">임곗값과 줄다리기</td>
<td style="text-align: left;">타이타닉에서도 임곗값을 바꾸면 3-2와 같은 방향으로 움직이고, 그 움직임은 항상 보장되는가?</td>
</tr>
<tr>
<td style="text-align: left;">4</td>
<td style="text-align: left;">확률 읽기</td>
<td style="text-align: left;">확률 0.97은 '97% 맞다'는 뜻인가? 확률 분포 · z · 피처 수는 서로 어떻게 이어지는가?</td>
</tr>
<tr>
<td style="text-align: left;">5</td>
<td style="text-align: left;">계수와 인코딩</td>
<td style="text-align: left;">모델의 판단 근거를 계수로 설명할 수 있는가? 나이 · 인코딩은 어떻게 다뤄야 하는가?</td>
</tr>
<tr>
<td style="text-align: left;">6</td>
<td style="text-align: left;">종합 실습</td>
<td style="text-align: left;">valid에서 고른 것을 모두 고정하고 test를 처음 열면?</td>
</tr>
<tr>
<td style="text-align: left;">—</td>
<td style="text-align: left;">마무리 · 부록</td>
<td style="text-align: left;">오늘 배운 것 · 기억 확인 · 오답 노트 · 면접 · 요약표 · 복습일 · 원본 대응표 · 전체 코드</td>
</tr>
</tbody>
</table>

---
## 0. 준비

원본 폴더의 `train.csv`를 읽음 (원본은 같은 폴더에 두라고 안내). **셀은 위에서부터 차례로 실행**.

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">항목</th>
<th style="text-align: left;">설정</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">라이브러리</td>
<td style="text-align: left;">numpy · pandas · scikit-learn (원본의 plotly 그림은 정적 그림으로)</td>
</tr>
<tr>
<td style="text-align: left;">seed</td>
<td style="text-align: left;">분할 <code>random_state=42</code> · 로지스틱 42 (원본과 같음)</td>
</tr>
<tr>
<td style="text-align: left;">데이터</td>
<td style="text-align: left;"><code>../03_전처리_평가지표_Titanic/train.csv</code> · 891명 · 생존 38.4%</td>
</tr>
<tr>
<td style="text-align: left;">원본 그림</td>
<td style="text-align: left;">슬라이드 스크린샷 4장 → 💡 · 비교 표의 글로</td>
</tr>
</tbody>
</table>

In [1]:
# 준비: 타이타닉 891명 읽기 (원본 폴더의 train.csv)
import numpy as np                                                           # 배열 · 수식
import pandas as pd                                                          # 표
from sklearn.compose import ColumnTransformer                                # 열마다 다른 전처리
from sklearn.impute import SimpleImputer                                     # 결측 채우기
from sklearn.linear_model import LogisticRegression                          # 오늘의 모델
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score          # 지표
from sklearn.metrics import confusion_matrix, precision_score, recall_score  # 지표
from sklearn.model_selection import train_test_split                         # 분할
from sklearn.pipeline import Pipeline                                        # 전처리 + 모델
from sklearn.preprocessing import OneHotEncoder, StandardScaler              # 인코딩 · 스케일링

CSV = "../03_전처리_평가지표_Titanic/train.csv"      # 원본 폴더 데이터 (복사하지 않음)
titanic_df = pd.read_csv(CSV)                        # 891명 × 12열
print(titanic_df.shape)                              # 행 · 열
cols = ["Survived", "Pclass", "Sex", "Age", "Fare"]  # 주요 열
print(titanic_df[cols].head(3).to_string())          # 앞 3명

(891, 12)
   Survived  Pclass     Sex   Age     Fare
0         0       3    male  22.0   7.2500
1         1       1  female  38.0  71.2833
2         1       3  female  26.0   7.9250


### 원본 목표 · 분류란

- **핵심 질문** · "확률로 분류"한다는 건 무슨 뜻인가? → 이번엔 모델을 열어 그 확률이 **어디서 나온 숫자인지** 확인
- 3-2를 닫은 문장 · 숫자가 나왔다는 이유만으로 믿지 않음 · 모델을 믿기 전에 실험을 먼저 의심
- 목표 · 분류 · Sigmoid · 로지스틱 회귀 · 임곗값에 따른 정밀도 / 재현율 줄다리기

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">분야</th>
<th style="text-align: left;">분류 예시</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">금융 · 이메일</td>
<td style="text-align: left;">사기 거래인가? · 스팸인가?</td>
</tr>
<tr>
<td style="text-align: left;">의료 · 이미지</td>
<td style="text-align: left;">암 환자인가? · 고양이인가 강아지인가?</td>
</tr>
<tr>
<td style="text-align: left;">마케팅</td>
<td style="text-align: left;">이 고객이 이탈할 것인가?</td>
</tr>
</tbody>
</table>

**원본 힌트**

- 4회차 핵심 문장 · 분류 모델은 각각 다른 방식으로 경계를 긋는다 — 확률(로지스틱 4-1) · 거리(KNN 4-2) · 규칙(트리)
- 오늘의 규칙 · 노트북이 끝날 때까지 test는 한 번도 안 나오고, 맨 마지막(6장)에서 딱 한 번

---
## 1. Sigmoid

> **이 장의 흐름**
>
> - 점수 z는 -∞ ~ +∞ → Sigmoid로 0 ~ 1
> - z = 0이 확률 0.5인 결정 경계, 가운데는 민감 · 양끝은 둔감

<img src="images/pfm5_sigmoid.png" alt="Sigmoid · z = 0 ⇔ 0.5 · 가운데는 민감 · 양끝은 둔감" width="602">

**파이프라인 위치** · ④ 모델 — 출력층

### 1-1. `A` Sigmoid 함수 · `1 / (1 + np.exp(-z))` <sub>원본 셀 3 ~ 5</sub>

> 🎯 **z = -10 · 0 · 2.3 · +10은 각각 확률 몇이 되는가?**

💡 **쉽게 말하면**

- **시험 점수(z)를 합격 확률로** 바꾸는 변환기.
- z = 0이면 반반, 멀어질수록 0이나 1에 붙음.

**실행 흐름**

1. sigmoid 함수 정의
2. z 값 6개 변환
3. z 1 차이가 확률을 얼마나 바꾸나

**먼저 예측** · z = 2.3의 확률은? — 답은 코드 실행 결과 아래에

In [2]:
def sigmoid(z):                  # 점수 → 확률
    return 1 / (1 + np.exp(-z))  # 원본 수식


for z in [-10, -2, 0, 2, 2.3, 10]:                           # 원본 표의 값 + 예시 2.3
    print(f"z = {z:>5} → σ(z) = {sigmoid(z):.3f}")           # 0 ~ 1
print("가운데 z 0 → 1:", round(sigmoid(1) - sigmoid(0), 3))  # 민감
print("끝쪽 z 5 → 6:", round(sigmoid(6) - sigmoid(5), 3))    # 둔감

z =   -10 → σ(z) = 0.000
z =    -2 → σ(z) = 0.119
z =     0 → σ(z) = 0.500
z =     2 → σ(z) = 0.881
z =   2.3 → σ(z) = 0.909
z =    10 → σ(z) = 1.000
가운데 z 0 → 1: 0.231
끝쪽 z 5 → 6: 0.004


**결과** · 예측 정답은 0.909

- `z =   -10 → σ(z) = 0.000` — 거의 확실히 음성
- `z =     0 → σ(z) = 0.500` — 반반 · 결정 경계
- `z =   2.3 → σ(z) = 0.909` — 0.909 ≥ 0.5 → 양성(생존)
- `가운데 z 0 → 1: 0.231` — 경계 근처는 z 1 차이로 확률이 0.231 변함 (민감)
- `끝쪽 z 5 → 6: 0.004` — 양끝은 거의 안 변함 (둔감)

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">z (점수)</td>
<td style="text-align: left;">-∞ ~ +∞ · <code>decision_function</code></td>
<td style="text-align: left;">순위 · 경계로부터의 방향</td>
</tr>
<tr>
<td style="text-align: left;">σ(z) (확률)</td>
<td style="text-align: left;">0 ~ 1 · <code>predict_proba</code></td>
<td style="text-align: left;">임곗값 · 사람에게 설명</td>
</tr>
<tr>
<td style="text-align: left;">z = 0 ⇔ 0.5</td>
<td style="text-align: left;">기본 결정 경계</td>
<td style="text-align: left;">기본 임곗값 0.5와 같은 뜻</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - σ(z)가 0.909여도 '91% 맞다'는 뜻은 아님 (4-1)

> ✅ **한 줄 정리** · Sigmoid는 점수 z를 0 ~ 1 확률로 바꾸며, z = 0이 확률 0.5인 결정 경계다.

**딥러닝 코드에서는**

- `torch.sigmoid(logit)` — 이진 분류 출력층
- 학습은 logit 그대로 `BCEWithLogitsLoss`

---
## 2. 로지스틱 회귀 학습

> **이 장의 흐름**
>
> - test를 먼저 봉인하는 3분할 → 열마다 다른 전처리 Pipeline으로 학습 · 확률 출력
> - 교차 엔트로피 벌점 확인 → Dummy와 정확도 · AUC 비교

<img src="images/pfm5_pipeline.png" alt="타이타닉 Pipeline · 숫자 열과 글자 열을 따로 손질" width="763">

**파이프라인 위치** · ② 분할 → ③ 전처리 → ④ 학습

### 2-1. `A` 3분할과 Pipeline · `ColumnTransformer` · `Cabin1` <sub>원본 셀 6 ~ 9</sub>

> 🎯 **test를 먼저 떼어 낸 3분할과 열마다 다른 전처리는 어떻게 만드는가?**

💡 **쉽게 말하면**

- train = **교과서** · valid = **모의고사** · test = **수능** (마지막 1회).
- 숫자 열은 중앙값 + 표준화, 글자 열은 최빈값 + 원-핫.

**실행 흐름**

1. Cabin 첫 글자만(Cabin1) · 안 쓰는 열 제거
2. test 20% 봉인 → 나머지를 train · valid로
3. ColumnTransformer + 로지스틱 Pipeline

**먼저 예측** · train · valid · test는 각각 몇 명? — 답은 코드 실행 결과 아래에

In [3]:
y_all = titanic_df["Survived"]                                          # 정답
X_all = titanic_df.drop("Survived", axis=1).copy()                      # 문제
X_all["Cabin1"] = X_all["Cabin"].fillna("N").astype(str).str[:1]        # 선실 첫 글자 (없으면 N)
X_all = X_all.drop(["PassengerId", "Name", "Ticket", "Cabin"], axis=1)  # 쓰지 않는 열
print("데이터 크기:", X_all.shape, "· 생존 비율:", round(y_all.mean(), 4))  # 8열
X_dev, X_final_test, y_dev, y_final_test = train_test_split(            # 1단계: 수능 봉인
    X_all, y_all, test_size=0.2, stratify=y_all, random_state=42)
X_train, X_valid, y_train, y_valid = train_test_split(                  # 2단계: 교과서 · 모의고사
    X_dev, y_dev, test_size=0.25, stratify=y_dev, random_state=42)
print(f"train {len(X_train)} · valid {len(X_valid)} · test {len(X_final_test)}")  # 6 : 2 : 2
NUM = ["Pclass", "Age", "SibSp", "Parch", "Fare"]                       # 숫자 열
CAT = ["Sex", "Embarked", "Cabin1"]                                     # 글자 열


def make_lr_pipeline(feats):  # 고른 열만으로 Pipeline (4-3 · 5장에서도 씀)
    num = [f for f in feats if f in NUM]                                      # 고른 숫자 열
    cat = [f for f in feats if f in CAT]                                      # 고른 글자 열
    num_tf = Pipeline([("imputer", SimpleImputer(strategy="median")),         # 중앙값 + 표준화
                       ("scaler", StandardScaler())])
    cat_tf = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),  # 최빈값 + 원-핫
                       ("onehot", OneHotEncoder(handle_unknown="ignore"))])
    pre = ColumnTransformer([("num", num_tf, num), ("cat", cat_tf, cat)])     # 열마다 따로
    model = LogisticRegression(max_iter=5000, random_state=42)                # 원본 설정
    return Pipeline([("preprocess", pre), ("model", model)])                  # 전처리 + 모델


ALL = NUM + CAT             # 전체 8열
lr = make_lr_pipeline(ALL)  # 오늘의 기본 모델

데이터 크기: (891, 8) · 생존 비율: 0.3838
train 534 · valid 178 · test 179


**결과** · 예측 정답은 534 · 178 · 179

- `데이터 크기: (891, 8) · 생존 비율: 0.3838` — 891명 × 8열 · 생존 38.4%
- `train 534 · valid 178 · test 179` — test를 먼저 떼고, 남은 데이터를 3 : 1로

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">train (교과서)</td>
<td style="text-align: left;">모델 · 전처리를 fit</td>
<td style="text-align: left;">배울 때</td>
</tr>
<tr>
<td style="text-align: left;">valid (모의고사)</td>
<td style="text-align: left;">임곗값 · 피처 · 인코딩을 고름</td>
<td style="text-align: left;">선택할 때 (여러 번)</td>
</tr>
<tr>
<td style="text-align: left;">test (수능)</td>
<td style="text-align: left;">6장에서 딱 한 번</td>
<td style="text-align: left;">보고할 때</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 3-2와 같은 3분할 → 새 규칙이 아님 · 데이터와 모델이 바뀌어도 절차는 그대로
> - 오늘은 고를 것이 늘어남 (임곗값 + 피처 + 인코딩) → test를 지키는 것이 더 중요

> ✅ **한 줄 정리** · test를 먼저 봉인한 3분할과, 숫자 · 글자 열을 따로 손질하는 Pipeline으로 학습을 준비한다.

**딥러닝 코드에서는**

- `random_split` 3개 · 이미지 정규화 + 텍스트 토큰화처럼 입력마다 다른 전처리

### 2-2. `A` 학습 · `predict_proba` · 반복 횟수 · `n_iter_` <sub>원본 셀 10 ~ 15</sub>

> 🎯 **로지스틱은 0 / 1을 바로 찍는가, 확률을 먼저 만드는가? `max_iter=5000`은 5000번 도는가?**

💡 **쉽게 말하면**

- 로지스틱 = **직선(경계)을 긋고**, 선에서 떨어진 정도 z를 Sigmoid로 확률로.
- `max_iter`는 **상한** · 손실이 더 안 줄면 알아서 멈춤.

**실행 흐름**

1. train에 fit
2. valid 생존 확률 앞 10명
3. solver · 실제 반복 횟수
4. predict = 확률 ≥ 0.5

**먼저 예측** · 실제 반복 횟수는 5000에 가까울까? — 답은 코드 실행 결과 아래에

In [4]:
lr.fit(X_train, y_train)                                            # train에만 fit (전처리 포함)
y_proba = lr.predict_proba(X_valid)[:, 1]                           # 생존 확률 (열 1)
print("예측 확률 (생존) 앞 10명:", y_proba[:10].round(4).tolist())  # 원본 셀 12
lr_model = lr.named_steps["model"]                                  # Pipeline 안의 로지스틱
print(f"solver          : {lr_model.solver}")                       # 최적화 방법
print(f"max_iter (설정) : {lr_model.max_iter}")                     # 상한
print(f"n_iter_  (실제) : {lr_model.n_iter_[0]}")                   # 실제로 돈 횟수
same = (lr.predict(X_valid) == (y_proba >= 0.5)).all()              # predict = 확률 ≥ 0.5?
print("predict == (확률 ≥ 0.5):", same)                             # 내부에서 자동 비교

예측 확률 (생존) 앞 10명: [0.1093, 0.1176, 0.74, 0.1017, 0.084, 0.1017, 0.084, 0.9173, 0.0842, 0.557]
solver          : lbfgs
max_iter (설정) : 5000
n_iter_  (실제) : 32
predict == (확률 ≥ 0.5): True


**결과** · 예측 정답은 아님 — 32번에 수렴

- `예측 확률 (생존) 앞 10명:` — 0 / 1이 아니라 확률이 먼저 나옴
- `solver          : lbfgs` — 경사하강법이 아니라 준뉴턴법 계열 · 아이디어는 같음
- `n_iter_  (실제) : 32` — 5000은 상한일 뿐 · 손실이 안 줄면 멈춤
- `predict == (확률 ≥ 0.5): True` — `predict` = `predict_proba` → 0.5 비교

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>predict()</code></td>
<td style="text-align: left;">0 또는 1</td>
<td style="text-align: left;">기본 기준 0.5로 바로 답할 때</td>
</tr>
<tr>
<td style="text-align: left;"><code>predict_proba()[:, 1]</code></td>
<td style="text-align: left;">양성(생존) 확률</td>
<td style="text-align: left;">임곗값 · 순위 · AUC</td>
</tr>
<tr>
<td style="text-align: left;"><code>max_iter</code></td>
<td style="text-align: left;">반복 상한 · 수렴 경고 방지용</td>
<td style="text-align: left;">성능을 올리는 값이 아님</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 모델이 선을 직접 움직이는 게 아니라 `w`와 `b`를 조금씩 바꾼 결과로 선이 움직임
> - 반복 횟수 32는 이 데이터 · 이 전처리의 값 → 데이터가 바뀌면 달라짐

> ✅ **한 줄 정리** · 로지스틱은 확률을 먼저 만들고 `predict`는 그 확률을 0.5와 비교하며, `max_iter`는 반복 상한일 뿐이다.

**딥러닝 코드에서는**

- `logits = model(x)` → `torch.sigmoid` → 기준 비교 · epoch 수도 상한 + early stopping

### 2-3. `A` 교차 엔트로피 · 확신하고 틀리기 · `-np.log(1 - q)` <sub>원본 셀 11 · 16 ~ 18</sub>

> 🎯 **정답이 사망(0)인데 0.51로 틀린 것과 0.99로 틀린 것의 벌점은 몇 배 차이인가?**

💡 **쉽게 말하면**

- 정답 쪽 확률이 1에 가까우면 벌점이 작고, 0에 가까우면 **엄청 큼**.
- 그래서 로지스틱은 함부로 극단으로 밀지 않음 · 애매한 건 애매하게.

**실행 흐름**

1. 틀린 확률 4개의 벌점 -log(1 - q)
2. 0.99 vs 0.51 비율

**먼저 예측** · 0.99로 틀린 벌점은 0.51로 틀린 벌점의 몇 배? 2배? 10배? — 답은 코드 실행 결과 아래에

In [5]:
for q in [0.51, 0.7, 0.9, 0.99]:                               # 정답 0인데 낸 생존 확률
    loss = -np.log(1 - q)                                      # 정답(0) 쪽 확률의 -log
    print(f"정답=사망(0)인데 생존 확률 {q:.2f} 로 예측 → 벌점 {loss:.4f}")  # 원본 형식
ratio = -np.log(1 - 0.99) / -np.log(1 - 0.51)                  # 두 벌점의 비
print(f"0.99로 틀린 벌점 / 0.51로 틀린 벌점 = {ratio:.2f}배")  # 확신의 대가

정답=사망(0)인데 생존 확률 0.51 로 예측 → 벌점 0.7133
정답=사망(0)인데 생존 확률 0.70 로 예측 → 벌점 1.2040
정답=사망(0)인데 생존 확률 0.90 로 예측 → 벌점 2.3026
정답=사망(0)인데 생존 확률 0.99 로 예측 → 벌점 4.6052
0.99로 틀린 벌점 / 0.51로 틀린 벌점 = 6.46배


**결과** · 예측 정답은 6.46배

- `정답=사망(0)인데 생존 확률 0.51 로 예측 → 벌점 0.7133` — 거의 반반으로 틀림
- `정답=사망(0)인데 생존 확률 0.99 로 예측 → 벌점 4.6052` — 확신하고 틀림
- `0.99로 틀린 벌점 / 0.51로 틀린 벌점 = 6.46배` — 확률을 0.51 → 0.99로 밀었을 뿐인데 6배 넘게

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">정확도</td>
<td style="text-align: left;">맞았다 / 틀렸다만</td>
<td style="text-align: left;">결과 보고</td>
</tr>
<tr>
<td style="text-align: left;">교차 엔트로피</td>
<td style="text-align: left;">얼마나 확신했는지까지 채점</td>
<td style="text-align: left;"><strong>학습 손실</strong></td>
</tr>
<tr>
<td style="text-align: left;">L2 규제 (<code>C</code>)</td>
<td style="text-align: left;">계수가 너무 커지지 않게 누름 · 기본으로 켜짐</td>
<td style="text-align: left;">벌점 + 계수 크기를 같이 줄임</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 손실 식: −[y log p̂ + (1 − y) log(1 − p̂)] + 규제 → `C`가 작을수록 규제가 강함

> ✅ **한 줄 정리** · 교차 엔트로피는 확신하고 틀릴수록 벌점이 급격히 커져서(6.46배), 로지스틱이 애매한 것은 애매하게 두게 만든다.

**딥러닝 코드에서는**

- `nn.BCEWithLogitsLoss` = 같은 식 · 1회차의 '메인 재료 빠지면 극대노 미식가'

### 2-4. `A` 기준선과 비교 · 정확도는 같다 · `dummy_rule` · `roc_auc_score` <sub>원본 셀 19 ~ 22</sub>

> 🎯 **피처 8개 로지스틱의 valid 정확도는 '남자면 사망' 규칙보다 얼마나 높은가?**

💡 **쉽게 말하면**

- 비교 대상이 없으면 0.798이 좋은지 알 수 없음.
- 로지스틱이 나은 점은 더 많이 맞혀서가 아니라 **확률을 주기 때문**.

**실행 흐름**

1. ① 전부 사망 ② 남자 = 사망 규칙 ③ 로지스틱
2. 정확도 · AUC 표

**먼저 예측** · ②와 ③의 정확도 차이는 5%p? 10%p? — 답은 코드 실행 결과 아래에

In [6]:
def dummy_rule(X):                                        # 3-2 Part 1의 규칙
    return np.where(X["Sex"].to_numpy() == "male", 0, 1)  # 남자면 사망


acc_lr = accuracy_score(y_valid, (y_proba >= 0.5).astype(int))           # 기본 기준
acc_major = accuracy_score(y_valid, np.zeros(len(y_valid)))              # ① 전부 사망
acc_rule = accuracy_score(y_valid, dummy_rule(X_valid))                  # ② 규칙
auc_lr = round(roc_auc_score(y_valid, y_proba), 4)                       # ③ 순위 성능
rows = [["① 전부 사망 (다수 클래스)", acc_major, "-"],                   # 원본 셀 21의 표
        ["② 3-2 Dummy 규칙 (남자=사망)", acc_rule, "확률 없음"],
        ["③ 로지스틱 회귀 (피처 8개)", acc_lr, auc_lr]]
cols = ["모델", "valid Accuracy", "AUC"]                                 # 열 이름
print(pd.DataFrame(rows, columns=cols).round(4).to_string(index=False))  # 비교표

                    모델  valid Accuracy    AUC
      ① 전부 사망 (다수 클래스)          0.6180      -
② 3-2 Dummy 규칙 (남자=사망)          0.7978  확률 없음
     ③ 로지스틱 회귀 (피처 8개)          0.7978  0.857


**결과** · 예측 정답은 0 — 똑같음 (0.7978)

- `① 전부 사망 (다수 클래스)          0.6180      -` — 찍기 기준선
- `② 3-2 Dummy 규칙 (남자=사망)          0.7978  확률 없음` — 규칙 한 줄
- `③ 로지스틱 회귀 (피처 8개)          0.7978  0.857` — 정확도는 규칙과 같고 AUC 0.857

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">Dummy 규칙</td>
<td style="text-align: left;">정확도 같음 · 확률 불가 · AUC 잴 수 없음</td>
<td style="text-align: left;">넘어야 할 기준선</td>
</tr>
<tr>
<td style="text-align: left;">로지스틱</td>
<td style="text-align: left;">정확도 같음 · <code>predict_proba</code> · AUC 0.857</td>
<td style="text-align: left;">임곗값 · 순위 · 설명</td>
</tr>
<tr>
<td style="text-align: left;">정확도로 비교</td>
<td style="text-align: left;">두 모델을 구별 못 함</td>
<td style="text-align: left;">AUC · 지표 여러 개로</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 확률이 없으면 임곗값을 바꿀 수 없고, 순위를 매길 수 없고, AUC도 못 잼
> - 반대로 AUC가 높다고 반드시 좋은 것도 아님 → 무엇을 위해 쓸 모델인지에 달림

> ✅ **한 줄 정리** · 로지스틱이 Dummy보다 나은 것은 더 많이 맞혀서가 아니라, 정책에 맞게 임곗값을 고를 수 있는 확률을 주기 때문이다.

**딥러닝 코드에서는**

- 분류 모델 평가에 accuracy만 쓰지 않고 AUROC · 보정(calibration)까지

---
## 3. 임곗값과 줄다리기

> **이 장의 흐름**
>
> - 기준 0.3 · 0.5 · 0.7의 혼동행렬 → 규칙(F1 최대)을 먼저 선언하고 0.1 ~ 0.9 표에서 고름
> - 그다음 0.05 간격으로 촘촘히 → 정밀도가 오히려 내려가는 구간 찾기

<img src="images/pfm5_threshold.png" alt="타이타닉 valid · 임곗값별 정밀도 · 재현율 · F1" width="597">

**파이프라인 위치** · ⑤ 선택(valid)

### 3-1. `A` 기준 0.3 · 0.5 · 0.7 · `confusion_matrix` · `ravel()` <sub>원본 셀 23 ~ 25</sub>

> 🎯 **기준을 0.3 · 0.5 · 0.7로 바꾸면 FP와 FN은 어떻게 바뀌는가?**

💡 **쉽게 말하면**

- 낮추면 **많이 생존이라고 함** → 누락 ↓ · 오탐 ↑.
- 높이면 확실할 때만 → 오탐 ↓ · 누락 ↑.

**실행 흐름**

1. 기준마다 0 / 1
2. 정밀도 · 재현율
3. 혼동행렬 4칸

**먼저 예측** · 기준 0.7의 FN(살았는데 죽는다고 함)은 0.3보다 많을까? — 답은 코드 실행 결과 아래에

In [7]:
for thr in [0.3, 0.5, 0.7]:                                               # 원본 셀 24의 세 기준
    y_hat = (y_proba >= thr).astype(int)                                  # 확률 → 0 / 1
    tn, fp, fn, tp = confusion_matrix(y_valid, y_hat).ravel()             # 4칸
    p, r = precision_score(y_valid, y_hat), recall_score(y_valid, y_hat)  # 두 지표
    print(f"thr={thr} · P {p:.4f} · R {r:.4f}")                           # 두 지표
    print(f"  FP(오탐) {fp} · FN(누락) {fn} · TP {tp} · TN {tn}")         # 4칸

thr=0.3 · P 0.6706 · R 0.8382
  FP(오탐) 28 · FN(누락) 11 · TP 57 · TN 82
thr=0.5 · P 0.7286 · R 0.7500
  FP(오탐) 19 · FN(누락) 17 · TP 51 · TN 91
thr=0.7 · P 0.8140 · R 0.5147
  FP(오탐) 8 · FN(누락) 33 · TP 35 · TN 102


**결과** · 예측 정답은 많음 — 11명 → 33명

- 많이 잡음 · 오탐 28
  - `thr=0.3 · P 0.6706 · R 0.8382`
  - `FP(오탐) 28 · FN(누락) 11 · TP 57 · TN 82`
- 중간 균형
  - `thr=0.5 · P 0.7286 · R 0.7500`
  - `FP(오탐) 19 · FN(누락) 17 · TP 51 · TN 91`
- 확실할 때만 · 누락 33
  - `thr=0.7 · P 0.8140 · R 0.5147`
  - `FP(오탐) 8 · FN(누락) 33 · TP 35 · TN 102`

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">낮춤 (0.3)</td>
<td style="text-align: left;">재현율 ↑ · 정밀도 ↓</td>
<td style="text-align: left;">누락이 비쌀 때</td>
</tr>
<tr>
<td style="text-align: left;">기본 (0.5)</td>
<td style="text-align: left;">모델 기본값</td>
<td style="text-align: left;">기준을 정하기 전 출발점</td>
</tr>
<tr>
<td style="text-align: left;">높임 (0.7)</td>
<td style="text-align: left;">정밀도 ↑ · 재현율 ↓</td>
<td style="text-align: left;">오탐이 비쌀 때</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 타이타닉에서 FP(살 사람을 죽는다고 함) vs FN(죽을 사람을 산다고 함) 중 무엇이 비싼지는 쓰임새가 정함

> ✅ **한 줄 정리** · 기준을 낮추면 놓침이 줄고 오탐이 늘며, 높이면 그 반대로 움직인다.

**딥러닝 코드에서는**

- 검출 모델 confidence threshold — 같은 줄다리기

### 3-2. `A` 규칙 먼저 · valid 표에서 고르기 · `thr_df` · `idxmax()` <sub>원본 셀 26 ~ 29</sub>

> 🎯 **표를 보기 전에 'valid F1 최대'를 선언하면 어떤 기준이 고르게 되는가?**

💡 **쉽게 말하면**

- 표를 보고 기준을 정하면 **기준 자체를 데이터에 맞춘 것**.
- 그래서 규칙을 먼저 적고 표를 봄.

**실행 흐름**

1. 규칙 선언: valid F1 최대
2. 0.1 ~ 0.9 표
3. F1 최대 기준 확정

**먼저 예측** · F1 최대 기준은 0.3 · 0.5 · 0.7 중 어디쯤? — 답은 코드 실행 결과 아래에

In [8]:
thr_list = np.round(np.arange(0.1, 1.0, 0.1), 1)                                 # 후보 9개
rows = []                                                                        # 기준마다 지표
for thr in thr_list:                                                             # 원본 셀 27
    y_hat = (y_proba >= thr).astype(int)                                         # 0 / 1
    rows.append([thr, precision_score(y_valid, y_hat, zero_division=0),          # 세 지표
                 recall_score(y_valid, y_hat), f1_score(y_valid, y_hat)])
thr_df = pd.DataFrame(rows, columns=["Threshold", "Precision", "Recall", "F1"])  # 표
print(thr_df.round(4).to_string(index=False))                                    # 원본 표
chosen_thr = float(thr_df.loc[thr_df["F1"].idxmax(), "Threshold"])               # 선언한 규칙
print("valid F1 최대 기준:", chosen_thr)                                         # 6장에서 그대로 씀

 Threshold  Precision  Recall     F1
       0.1     0.4539  0.9412 0.6124
       0.2     0.6413  0.8676 0.7375
       0.3     0.6706  0.8382 0.7451
       0.4     0.7051  0.8088 0.7534
       0.5     0.7286  0.7500 0.7391
       0.6     0.8036  0.6618 0.7258
       0.7     0.8140  0.5147 0.6306
       0.8     0.8889  0.3529 0.5053
       0.9     0.9474  0.2647 0.4138


valid F1 최대 기준: 0.4


**결과** · 예측 정답은 0.4

- `0.4     0.7051  0.8088  0.7534` — F1 최대 행
- `0.5     0.7286  0.7500  0.7391` — 기본 기준은 F1이 조금 낮음
- `valid F1 최대 기준: 0.4` — 선언한 규칙대로 확정 → 6장 test에 그대로

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">놓치지 않는 것이 우선</td>
<td style="text-align: left;">재현율 요구조건을 먼저</td>
<td style="text-align: left;">암 · 사기</td>
</tr>
<tr>
<td style="text-align: left;">잘못 울리는 경보 비용이 큼</td>
<td style="text-align: left;">정밀도 · FP 비용</td>
<td style="text-align: left;">스팸</td>
</tr>
<tr>
<td style="text-align: left;">두 실수를 비슷하게</td>
<td style="text-align: left;">valid F1 최대</td>
<td style="text-align: left;">이번 실습</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 원본은 규칙(F1 최대)을 선언했지만 6장 최종 평가에서 0.5를 씀 → 이 노트북은 선언대로 0.4로 평가
> - 임곗값에 정답은 없음 · 업무 목표(정책)에 따라 고름

> ✅ **한 줄 정리** · 선택 규칙을 표보다 먼저 선언하고, valid 표에서 그 규칙으로 기준(0.4)을 확정한다.

**딥러닝 코드에서는**

- val에서 고른 기준을 설정 파일에 저장 → 배포 · test에서 그대로

### 3-3. `B` 줄다리기는 경향 · 촘촘히 보기 · `np.arange(0.05, 0.96, 0.05)` <sub>원본 셀 29 ~ 31</sub>

> 🎯 **임곗값을 올렸는데 정밀도가 오히려 내려가는 구간이 있는가?**

💡 **쉽게 말하면**

- 기준을 높이면 양성으로 찍는 사람이 **급격히 줄어듦**.
- 몇 명 안 남으면 한 명만 틀려도 비율이 크게 흔들림.

**실행 흐름**

1. 0.05 간격 19개 기준
2. 정밀도 · 재현율 · 양성 예측 수
3. 앞 기준보다 정밀도가 내려간 곳

**먼저 예측** · 내려가는 구간이 있다면 높은 쪽일까 낮은 쪽일까? — 답은 코드 실행 결과 아래에

In [9]:
fine = np.round(np.arange(0.05, 0.96, 0.05), 2)         # 촘촘한 기준 19개
prev, drops = None, []                                  # 앞 정밀도 · 내려간 곳
for t in fine:                                          # 기준을 올리며
    yh = (y_proba >= t).astype(int)                     # 0 / 1
    pr = precision_score(y_valid, yh, zero_division=0)  # 정밀도
    if prev is not None and pr < prev - 1e-9:           # 앞보다 낮아졌으면
        drops.append(f"thr={t} · {prev:.4f} → {pr:.4f} · 양성 예측 {int(yh.sum())}명")  # 기록
    prev = pr                                           # 다음 비교용
print(f"정밀도가 내려간 구간: {len(drops)}곳")          # 원본 셀 30
for d in drops:                                         # 하나씩
    print(" ", d)                                       # 위치 · 남은 양성 수

정밀도가 내려간 구간: 2곳
  thr=0.65 · 0.8036 → 0.7708 · 양성 예측 48명
  thr=0.95 · 0.9474 → 0.9167 · 양성 예측 12명


**결과** · 예측 정답은 있음 · 높은 쪽 (0.65 · 0.95)

- `정밀도가 내려간 구간: 2곳` — 줄다리기는 보장이 아님
- `thr=0.65 · 0.8036 → 0.7708 · 양성 예측 48명` — 48명 남은 상태에서 흔들림
- `thr=0.95 · 0.9474 → 0.9167 · 양성 예측 12명` — 12명뿐 → 1명이 비율을 크게 움직임

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">하면 안 되는 말</td>
<td style="text-align: left;">임곗값을 올리면 정밀도는 반드시 오름</td>
<td style="text-align: left;">쓰지 않기</td>
</tr>
<tr>
<td style="text-align: left;">해야 하는 말</td>
<td style="text-align: left;">보통은 오르지만 양성 예측이 몇 개 안 남으면 흔들림</td>
<td style="text-align: left;"><strong>정확한 표현</strong></td>
</tr>
<tr>
<td style="text-align: left;">데이터가 커지면</td>
<td style="text-align: left;">흔들림이 줄어듦</td>
<td style="text-align: left;">하락 구간의 위치는 split마다 다름</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 위 결과는 valid 178명 기준 → 하락 구간 위치를 일반화하지 않기

> ✅ **한 줄 정리** · 정밀도 · 재현율 트레이드오프는 일반적 경향이지 법칙이 아니며, 표본이 얇아지는 구간에서는 요동친다.

**딥러닝 코드에서는**

- PR 곡선이 오른쪽 끝에서 톱니 모양인 이유

---
## 4. 확률 읽기

> **이 장의 흐름**
>
> - 실제 라벨별 확률 분포 → 확신하고 틀린 사람 찾기
> - z = 선까지의 거리 × ‖w‖ 확인 → 피처를 늘리며 분포와 AUC 변화

<img src="images/pfm5_dist.png" alt="피처를 늘릴수록 사망 · 생존 확률 분포가 갈라짐" width="947">

**파이프라인 위치** · ⑥ 평가 — 확률의 의미

### 4-1. `A` 확신하고 틀린 사람 · `wrong` · `extreme` <sub>원본 셀 32 ~ 37</sub>

> 🎯 **확률이 0.9 이상인데 사망했거나 0.1 이하인데 생존한 사람이 있는가?**

💡 **쉽게 말하면**

- 사망 분포는 왼쪽, 생존 분포는 오른쪽에 몰릴수록 좋음 · 겹치는 구간이 헷갈리는 곳.
- 그런데 **확신하고도 틀릴 수 있음**.

**실행 흐름**

1. 확률 · 실제 · 예측 표
2. 틀린 사람 · 극단에서 틀린 사람
3. 구간별 사망 · 생존 수

**먼저 예측** · 0.9 이상으로 확신했는데 틀린 사람은 몇 명? — 답은 코드 실행 결과 아래에

In [10]:
chk = pd.DataFrame({"예측 확률": y_proba.round(3), "실제": y_valid.to_numpy()})  # 원본 셀 36
chk["예측 라벨"] = (chk["예측 확률"] >= 0.5).astype(int)                    # 기본 기준
wrong = chk[chk["예측 라벨"] != chk["실제"]]                                # 틀린 사람
extreme = wrong[(wrong["예측 확률"] >= 0.9) | (wrong["예측 확률"] <= 0.1)]  # 확신하고 틀림
print(f"전체 valid {len(chk)}명 중 틀린 예측: {len(wrong)}명")              # 원본 형식
print(f"그중 모델이 '확신'했는데 틀린 경우: {len(extreme)}명")              # 극단
print(extreme.sort_values("예측 확률").to_string())                         # 누구인지
bins = pd.cut(chk["예측 확률"], [0, 0.3, 0.7, 1.0], include_lowest=True)    # 낮음 · 중간 · 높음
by_bin = chk.groupby(bins, observed=True)["실제"].agg(["size", "mean"])     # 구간별 인원 · 생존률
print(by_bin.round(3).to_string())  # 확률이 높아도 전부 생존은 아님

전체 valid 178명 중 틀린 예측: 36명
그중 모델이 '확신'했는데 틀린 경우: 5명
     예측 확률  실제  예측 라벨
108  0.075   1      0
54   0.089   1      0
105  0.100   1      0
162  0.100   1      0
43   0.956   0      1
               size   mean
예측 확률                     
(-0.001, 0.3]    93  0.118
(0.3, 0.7]       42  0.524
(0.7, 1.0]       43  0.814


**결과** · 예측 정답은 1명 (0.956) · 반대쪽 0.1 이하 생존도 4명

- `전체 valid 178명 중 틀린 예측: 36명` — 정확도 0.798의 나머지
- `그중 모델이 '확신'했는데 틀린 경우: 5명` — 극단에서 틀린 사람 5명
- `43   0.956   0      1` — 생존 확률 0.956인데 실제 사망
- `(0.7, 1.0]       43  0.814` — 0.7 넘게 준 43명 중 81.4%만 생존 → 확률 ≠ 정답 보장

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">'97% 확신한다'로 읽기</td>
<td style="text-align: left;">확률을 정답률로 착각</td>
<td style="text-align: left;">쓰지 않기</td>
</tr>
<tr>
<td style="text-align: left;">'생존 쪽으로 강하게 판단했다'로 읽기</td>
<td style="text-align: left;">모델의 기울기</td>
<td style="text-align: left;"><strong>올바른 읽기</strong></td>
</tr>
<tr>
<td style="text-align: left;">보정(calibration)</td>
<td style="text-align: left;">확률이 실제 빈도와 맞는지 검사</td>
<td style="text-align: left;">확률 숫자를 그대로 쓰려면</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 원본 셀 37은 'valid 179명 기준'이라 적었지만 valid는 178명 (179명은 test)
> - 표본이 작아 '몇 명'이라는 숫자 자체를 일반화하면 안 됨

> ✅ **한 줄 정리** · `predict_proba` 0.97은 '97% 맞다'가 아니라 '생존 쪽으로 강하게 기울었다'는 뜻이며, 모델은 확신하면서도 틀린다.

**딥러닝 코드에서는**

- 딥러닝 분류기는 과신(overconfidence)하기 쉬움 → temperature scaling으로 보정

### 4-2. `B` z = 선까지의 거리 × ‖w‖ · `decision_function` · `np.linalg.norm` <sub>원본 셀 11 · 41 ~ 43</sub>

> 🎯 **점수 z는 경계선까지의 거리와 어떤 관계인가?**

💡 **쉽게 말하면**

- 선 위면 z = 0 → 확률 0.5 · 멀어질수록 z가 커짐.
- 정확히는 z = 거리 × ‖w‖ → **순서는 같고 배율만 다름**.

**실행 흐름**

1. 전처리된 valid 입력
2. decision_function으로 z
3. ‖w‖로 나눠 거리 · σ(z)와 확률 비교

**먼저 예측** · z ÷ ‖w‖는 z와 순위가 같을까? — 답은 코드 실행 결과 아래에

In [11]:
Xt = lr.named_steps["preprocess"].transform(X_valid)                  # 전처리된 입력
Xt = Xt.toarray() if hasattr(Xt, "toarray") else Xt                   # 희소 → 배열
z = lr_model.decision_function(Xt)                                    # 점수 z
w_norm = np.linalg.norm(lr_model.coef_[0])                            # 계수 벡터의 길이
dist = z / w_norm                                                     # 선까지의 실제 거리
print(f"‖w‖ = {w_norm:.4f}")                                          # 배율
print(f"z    범위 : {z.min():.3f} ~ {z.max():.3f}")                   # 점수
print(f"거리  범위 : {dist.min():.3f} ~ {dist.max():.3f}")            # 거리
print("σ(z) = predict_proba:", np.allclose(sigmoid(z), y_proba))      # 확률은 z로 만듦
same_auc = roc_auc_score(y_valid, z) == roc_auc_score(y_valid, dist)  # 순위 지표 비교
print("AUC(z) = AUC(거리):", same_auc)                                # 비례하면 순위가 같음

‖w‖ = 2.5839
z    범위 : -3.304 ~ 4.086
거리  범위 : -1.279 ~ 1.581
σ(z) = predict_proba: True
AUC(z) = AUC(거리): True


**결과** · 예측 정답은 같음 — AUC도 같음

- `‖w‖ = 2.5839` — z는 거리의 2.58배
- `z    범위 : -3.304 ~ 4.086` — 점수 범위
- `거리  범위 : -1.279 ~ 1.581` — z ÷ 2.5839
- `σ(z) = predict_proba: True` — 확률은 거리가 아니라 z로 만듦
- `AUC(z) = AUC(거리): True` — 비례 관계라 순위 기반 지표는 같음

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">거리</td>
<td style="text-align: left;">기하학적 거리 = z ÷ ‖w‖</td>
<td style="text-align: left;">경계에서 얼마나 떨어졌나</td>
</tr>
<tr>
<td style="text-align: left;">z</td>
<td style="text-align: left;">Sigmoid에 들어가는 값</td>
<td style="text-align: left;">확률 계산</td>
</tr>
<tr>
<td style="text-align: left;">‖w‖</td>
<td style="text-align: left;">배율 · 크면 같은 거리도 확률이 극단으로</td>
<td style="text-align: left;">규제 <code>C</code>와 관련 (오늘 범위 밖)</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - '선과의 거리가 곧 확신'은 방향은 맞지만, Sigmoid에 들어가는 것은 z이고 배율이 ‖w‖

> ✅ **한 줄 정리** · 점수 z는 경계선까지의 거리에 ‖w‖를 곱한 값이라 순위는 같고, 확률은 z를 Sigmoid에 넣어 만든다.

**딥러닝 코드에서는**

- logit이 커지면 softmax가 한쪽으로 쏠림 → weight decay가 ‖w‖를 눌러 과신을 줄임

### 4-3. `A` 피처를 늘리면 · AUC · `make_lr_pipeline` · `roc_auc_score` <sub>원본 셀 38 ~ 40 · 44 · 51 · 52</sub>

> 🎯 **Pclass만 → Sex만 → Pclass + Sex → 전체로 늘리면 AUC는 어떻게 변하는가?**

💡 **쉽게 말하면**

- 피처가 강할수록 사망에는 낮은, 생존에는 높은 확률 → **두 분포가 갈라짐**.
- AUC = 생존자가 사망자보다 더 높은 점수를 받을 확률.

**실행 흐름**

1. 피처 조합 4개로 Pipeline
2. valid AUC · 정확도
3. Pclass만일 때 나오는 확률 값 종류

**먼저 예측** · ①→② 변화와 ②→③ 변화 중 어느 쪽이 클까? — 답은 코드 실행 결과 아래에

In [12]:
configs = [("Pclass만", ["Pclass"]), ("Sex만", ["Sex"]),                      # 원본 셀 51의 조합
           ("Pclass + Sex", ["Pclass", "Sex"]), ("전체 피처", ALL)]
rows, probs = [], {}                                                          # 표 · 확률 보관
for name, feats in configs:                                                   # 조합마다
    m = make_lr_pipeline(feats).fit(X_train, y_train)                         # 같은 절차
    p = m.predict_proba(X_valid)[:, 1]                                        # valid 확률
    probs[name] = p                                                           # 몇 종류인지 볼 때 씀
    acc_ = accuracy_score(y_valid, p >= 0.5)                                  # 기본 기준 정확도
    rows.append([name, len(feats), roc_auc_score(y_valid, p), acc_])          # 지표
cols = ["모델", "원본 컬럼 수", "AUC", "Accuracy"]                            # 원본 열 이름
print(pd.DataFrame(rows, columns=cols).round(3).to_string(index=False))       # 원본 표
print("Pclass만 확률 값:", sorted(set(probs["Pclass만"].round(3).tolist())))  # 몇 종류?

          모델  원본 컬럼 수   AUC  Accuracy
     Pclass만        1 0.702     0.669
        Sex만        1 0.780     0.798
Pclass + Sex        2 0.838     0.798
       전체 피처        8 0.857     0.798
Pclass만 확률 값: [0.248, 0.437, 0.647]


**결과** · 예측 정답은 ①→②가 큼 (0.702 → 0.838)

- `Pclass만 1 0.702 0.669` — 3등석에도 생존자 · 1등석에도 사망자 → 분포가 겹침
- `Sex만 1 0.780 0.798` — Sex 하나가 Pclass보다 강함 · 정확도는 Dummy와 같음
- `Pclass + Sex 2 0.838 0.798` — 두 개만 합쳐도 전체에 근접
- `전체 피처 8 0.857 0.798` — 나머지는 예외 케이스를 미세 조정
- `Pclass만 확률 값: [0.248, 0.437, 0.647]` — Pclass 1 · 2 · 3 세 값뿐 → 확률도 3개 · 분포가 뭉텅이로 겹침

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">Sex</td>
<td style="text-align: left;">가장 강한 신호 · 큰 틀을 결정</td>
<td style="text-align: left;">'Women and children first'</td>
</tr>
<tr>
<td style="text-align: left;">Pclass</td>
<td style="text-align: left;">보조 신호 · 같은 성별 안에서 추가 구분</td>
<td style="text-align: left;">등급 숫자가 커질수록 생존 ↓</td>
</tr>
<tr>
<td style="text-align: left;">나머지 (Age · Fare …)</td>
<td style="text-align: left;">예외 케이스 처리</td>
<td style="text-align: left;">미세 조정</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 원본 셀 44는 세 확률을 0.651 · 0.438 · 0.245로 적었지만 이 환경 값은 0.647 · 0.437 · 0.248 (끝자리만 다름)
> - '원본 컬럼 수 8'은 인코딩 전 개수 → 원-핫으로 펼치면 모델이 보는 입력은 19개 (5-3)
> - 피처를 많이 넣는다고 무조건 좋은 게 아니라 강한 신호 1 ~ 2개가 대부분을 결정할 수 있음

> ✅ **한 줄 정리** · 강한 피처가 들어갈수록 사망 · 생존의 확률 분포가 갈라지고, 그것이 곧 AUC가 높다는 뜻이다.

**딥러닝 코드에서는**

- 사전학습 백본 = 강한 피처 · 분류 헤드만 바꿔도 AUC가 크게 오름

---
## 5. 계수와 인코딩

> **이 장의 흐름**
>
> - '아이 먼저'가 데이터에 보이는지 → Pclass + Sex 모델의 계수와 6조합 확률
> - 그다음 원-핫 vs 정수 라벨 인코딩을 같은 절차로 비교

<img src="images/pfm5_coef.png" alt="Pclass + Sex 6조합 생존 확률 · 3등석 여성 > 1등석 남성" width="602">

**파이프라인 위치** · ④ 모델 해석 · ③ 전처리 선택

### 5-1. `B` 아이는 먼저였나 · `groupby` · `agg` <sub>원본 셀 45 ~ 47</sub>

> 🎯 **'Women and children first'의 아이 부분도 데이터에 보이는가?**

💡 **쉽게 말하면**

- 남자는 아이가 성인보다 3배 넘게 생존 · 여자는 오히려 아이가 낮음.
- 성별에 따라 나이의 효과가 다름 = **상호작용** → Age 숫자 하나로는 표현 못 함.

**실행 흐름**

1. Age 결측 제외 714명
2. 12세 이하 / 13세 이상
3. 성별 × 연령대 생존률

**먼저 예측** · 여자 아이와 여자 성인 중 어느 쪽 생존률이 높을까? — 답은 코드 실행 결과 아래에

In [13]:
age_df = titanic_df.dropna(subset=["Age"]).copy()                            # 나이를 아는 714명
age_df["연령대"] = np.where(age_df["Age"] <= 12, "아이(0~12)", "성인(13+)")  # 두 그룹
grp = age_df.groupby(["Sex", "연령대"])["Survived"]                          # 성별 × 연령대
tbl = grp.agg(인원="size", 생존률="mean")                                    # 인원 · 생존률
print(tbl.round(3).to_string())                                              # 원본 셀 46
print("나이 아는 인원:", len(age_df), "· 결측:", titanic_df["Age"].isna().sum())  # 대상 확인

                  인원    생존률
Sex    연령대                 
female 성인(13+)   229  0.777
       아이(0~12)   32  0.594
male   성인(13+)   416  0.173
       아이(0~12)   37  0.568
나이 아는 인원: 714 · 결측: 177


**결과** · 예측 정답은 여자 성인 (0.777 > 0.594)

- `female 성인(13+)   229  0.777` — 여자 성인이 가장 높음
- `아이(0~12)   32  0.594` — 여자 아이는 오히려 낮음
- `male   성인(13+)   416  0.173` — 남자 성인이 가장 낮음
- `아이(0~12)   37  0.568` — 남자 아이는 성인의 3배 넘게 생존
- `나이 아는 인원: 714 · 결측: 177` — 결측을 중앙값으로 채운 모델과는 대상이 다름

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">Age 숫자 그대로</td>
<td style="text-align: left;">1살당 같은 방향 · 직선 관계만</td>
<td style="text-align: left;">효과가 단조로울 때</td>
</tr>
<tr>
<td style="text-align: left;"><code>is_child</code> 같은 열 추가</td>
<td style="text-align: left;">아이 여부를 직접 알려 줌</td>
<td style="text-align: left;">구간에 의미가 있을 때</td>
</tr>
<tr>
<td style="text-align: left;">성별 × 아이 상호작용 열</td>
<td style="text-align: left;">성별마다 다른 나이 효과</td>
<td style="text-align: left;">이번 데이터 패턴</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 모델은 역사적 규칙이 아니라 **데이터에 남은 패턴**을 배움 → 이 숫자는 관찰된 비율

> ✅ **한 줄 정리** · '아이 먼저'는 남성 안에서만 뚜렷해서, 나이 숫자 하나로는 표현할 수 없는 상호작용이다.

**딥러닝 코드에서는**

- 신경망은 은닉층이 상호작용을 스스로 만듦 · 선형 모델은 열을 직접 만들어 줘야 함

### 5-2. `A` 계수 · 6조합 확률 · `coef_` · `get_feature_names_out` <sub>원본 셀 48 ~ 50</sub>

> 🎯 **Sex와 Pclass의 계수 부호는? 3등석 여성과 1등석 남성 중 누가 높은가?**

💡 **쉽게 말하면**

- 여성이면 **+ 가중치** → z ↑ · 남성이면 **− 가중치** → z ↓.
- 로지스틱은 계수를 꺼내 근거를 댈 수 있음 (KNN은 못 함).

**실행 흐름**

1. Pclass + Sex 모델
2. 피처 이름 · 계수 · 절편
3. 6조합 확률 표

**먼저 예측** · 3등석 여성 vs 1등석 남성, 누구의 생존 확률이 높을까? — 답은 코드 실행 결과 아래에

In [14]:
m_ps = make_lr_pipeline(["Pclass", "Sex"]).fit(X_train, y_train)     # 피처 2개
names = m_ps.named_steps["preprocess"].get_feature_names_out()       # 전처리 후 열 이름
coefs = m_ps.named_steps["model"].coef_[0]                           # 계수 w
coef_df = pd.DataFrame({"피처": names, "계수 w": coefs.round(3)})    # 열 이름 + 계수
print(coef_df.to_string(index=False))                                # 부호 확인
print(f"절편 b: {m_ps.named_steps['model'].intercept_[0]:.3f}")      # 기준점
combo = X_valid[["Pclass", "Sex"]].drop_duplicates()                 # 6조합
combo["예측 확률(생존)"] = m_ps.predict_proba(combo)[:, 1].round(3)  # 조합별 확률
print(combo.sort_values(["Sex", "Pclass"]).to_string(index=False))   # 원본 셀 49

             피처   계수 w
    num__Pclass -0.826
cat__Sex_female  1.307
  cat__Sex_male -1.306
절편 b: -0.248
 Pclass    Sex  예측 확률(생존)
      1 female      0.913
      2 female      0.796
      3 female      0.592
      1   male      0.436
      2   male      0.223
      3   male      0.096


**결과** · 예측 정답은 3등석 여성 (0.592 > 0.436)

- `num__Pclass -0.826` — 음수 → 등급 숫자가 커질수록(3등석) 생존 ↓
- `cat__Sex_female  1.307` — 여성 + 가중치
- `cat__Sex_male -1.306` — 남성 − 가중치
- `3 female            0.592` — 3등석 여성
- `1   male            0.436` — 1등석 남성 → 등급을 두 단계 낮춰도 성별을 못 뒤집음

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">계수 부호</td>
<td style="text-align: left;">+ 생존 쪽 · − 사망 쪽</td>
<td style="text-align: left;">방향 설명</td>
</tr>
<tr>
<td style="text-align: left;">계수 크기 비교</td>
<td style="text-align: left;">표준화 숫자 vs 0 / 1 원-핫 → 단위가 다름</td>
<td style="text-align: left;">크기만으로 중요도 단정 금지</td>
</tr>
<tr>
<td style="text-align: left;">조합별 확률 표</td>
<td style="text-align: left;">실측으로 비교</td>
<td style="text-align: left;"><strong>결론은 여기서</strong></td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - 원본 셀 50은 |1.221| > |0.802|라고 적었지만 실제 출력 계수는 1.307 · -0.826 (예전 실행 값)
> - Pclass는 표준화(1 = 1 표준편차), Sex는 0 / 1 → 계수 크기로 'Sex가 더 중요'라고 단정하지 않기

> ✅ **한 줄 정리** · 계수 부호로 방향을 설명하고, 크기 비교 대신 조합별 확률로 '3등석 여성 > 1등석 남성'을 확인한다.

**딥러닝 코드에서는**

- 신경망은 계수가 수백만 개 → SHAP · Grad-CAM 같은 별도 설명 도구가 필요

### 5-3. `B` 원-핫 vs 정수 라벨 · `OrdinalEncoder` · `handle_unknown` <sub>원본 셀 52 ~ 54</sub>

> 🎯 **범주형을 정수 라벨로 바꾸면 원-핫보다 성능이 얼마나 나쁜가?**

💡 **쉽게 말하면**

- 차이가 **생각보다 작음** · AUC는 사실상 같음.
- 원-핫을 쓰는 이유는 성능이 아니라 **의미** (없던 순서를 만들지 않음).

**실행 흐름**

1. 인코더만 다른 Pipeline 두 개
2. 모델이 보는 차원 · AUC · 정확도

**먼저 예측** · 원-핫이 확 좋을까? — 답은 코드 실행 결과 아래에

In [15]:
from sklearn.preprocessing import OrdinalEncoder  # 정수 라벨


def make_pipeline_with(enc):                                               # 인코더만 바꾼 Pipeline
    num_tf = Pipeline([("imputer", SimpleImputer(strategy="median")),      # 숫자는 같음
                       ("scaler", StandardScaler())])
    cat_tf = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),  # 인코더만 다름
                       ("enc", enc)])
    pre = ColumnTransformer([("num", num_tf, NUM), ("cat", cat_tf, CAT)])  # 같은 열 구성
    model = LogisticRegression(max_iter=5000, random_state=42)             # 같은 모델
    return Pipeline([("preprocess", pre), ("model", model)])               # 전처리 + 모델


ord_enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)   # 처음 본 값은 -1
encs = [("정수 라벨 (예전 방식)", ord_enc),                                      # 비교할 두 인코더
        ("OneHot (지금 방식)", OneHotEncoder(handle_unknown="ignore"))]
for label, enc in encs:                                                          # 같은 절차
    mm = make_pipeline_with(enc).fit(X_train, y_train)                           # train에 fit
    pp = mm.predict_proba(X_valid)[:, 1]                                         # valid 확률
    dim = mm.named_steps["preprocess"].transform(X_train).shape[1]               # 모델이 보는 차원
    auc_, acc_ = roc_auc_score(y_valid, pp), accuracy_score(y_valid, pp >= 0.5)  # 두 지표
    print(f"{label} · 차원 {dim} · AUC {auc_:.4f} · Acc {acc_:.4f}")             # 원본 셀 53의 표

정수 라벨 (예전 방식) · 차원 8 · AUC 0.8566 · Acc 0.7809
OneHot (지금 방식) · 차원 19 · AUC 0.8570 · Acc 0.7978


**결과** · 예측 정답은 아님 — AUC 0.8566 vs 0.8570

- `정수 라벨 (예전 방식) · 차원 8 · AUC 0.8566 · Acc 0.7809` — Embarked C 0 · Q 1 · S 2처럼 없던 순서
- `OneHot (지금 방식) · 차원 19 · AUC 0.8570 · Acc 0.7978` — 열 8개가 19개로 · AUC는 사실상 같음

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">정수 라벨</td>
<td style="text-align: left;">차원 그대로 · 가짜 순서</td>
<td style="text-align: left;">이번엔 우연히 해가 작았음</td>
</tr>
<tr>
<td style="text-align: left;">원-핫</td>
<td style="text-align: left;">차원 증가 · 순서 없음</td>
<td style="text-align: left;"><strong>명목형 기본</strong></td>
</tr>
<tr>
<td style="text-align: left;">범주가 많을 때</td>
<td style="text-align: left;">가짜 순서의 해가 커질 수 있음</td>
<td style="text-align: left;">원-핫 · 임베딩</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - split을 바꾸면 두 방식의 순위가 뒤바뀔 수도 있음 → 차이가 작다는 것이 요점
> - 전처리를 제대로 하는 이유는 점수가 아니라 **모델이 데이터를 오해하지 않게** 하려는 것

> ✅ **한 줄 정리** · 원-핫과 정수 라벨의 성능 차이는 작았지만, 원-핫은 없던 순서를 만들지 않으므로 명목형의 기본이다.

**딥러닝 코드에서는**

- 범주 → 임베딩: 순서도 차원 폭발도 없이 학습되는 벡터

---
## 6. 종합 실습 · 수능 보러 가기 · test 1회

> **이 장의 흐름**
>
> - 1 ~ 5장에서 valid로 고른 것(전체 피처 · 원-핫 · 기준 0.4)을 고정
> - test를 처음이자 마지막으로 열어 AUC · 정확도 · F1을 보고

<img src="images/pfm5_practice.png" alt="valid(고를 때 쓴 점수) vs test(보고할 점수)" width="605">

**파이프라인 위치** · ⑤ 선택 고정 → ⑥ test 1회

> 🎯 **valid에서 고른 설정으로 test를 한 번 평가하면 valid보다 높을까 낮을까?**

💡 **쉽게 말하면**

- valid 점수는 **그걸 보고 골랐기 때문에** 조금 유리.
- test 점수가 더 정직 → 보고는 test.

**실행 흐름**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">단계</th>
<th style="text-align: left;">하는 일</th>
<th style="text-align: left;">쓰는 문법</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1</td>
<td style="text-align: left;">고른 것 정리 (전처리 · 피처 · 인코딩 · 기준)</td>
<td style="text-align: left;">2-1 · 3-2 · 4-3 · 5-3</td>
</tr>
<tr>
<td style="text-align: left;">2</td>
<td style="text-align: left;">train에 최종 모델 fit</td>
<td style="text-align: left;">2-2</td>
</tr>
<tr>
<td style="text-align: left;">3</td>
<td style="text-align: left;">valid · test 확률</td>
<td style="text-align: left;">2-2</td>
</tr>
<tr>
<td style="text-align: left;">4</td>
<td style="text-align: left;">기준 0.4로 test 1회 · 원본처럼 0.5도 함께</td>
<td style="text-align: left;">3-2</td>
</tr>
</tbody>
</table>

**먼저 예측** · test AUC는 valid 0.857보다 높을까 낮을까? — 답은 코드 실행 결과 아래에

In [16]:
# 1 · 2단계: valid에서 고른 것을 고정하고 train에 fit
choices = {"전처리": "숫자 중앙값 + 표준화 · 글자 최빈값 + 원-핫",  # 전부 valid로 고름
           "피처": "전체 8열", "인코딩": "OneHot (5-3)", "임곗값": chosen_thr}
print(choices)                                                      # 고정한 선택
final_model = make_lr_pipeline(ALL).fit(X_train, y_train)           # 같은 설정 · 같은 train


# 3 · 4단계: valid 점수(참고)와 test 1회
rows = []                                                                # 세트별 점수
sets = [("valid (모의고사)", X_valid, y_valid),                          # 참고용
        ("test (수능)", X_final_test, y_final_test)]                     # 처음이자 마지막
for name, X_, y_ in sets:                                                # 같은 지표로
    p_ = final_model.predict_proba(X_)[:, 1]                             # 확률
    hat = p_ >= chosen_thr                                               # 선언한 기준 0.4
    rows.append([name, len(y_), roc_auc_score(y_, p_), accuracy_score(y_, hat),  # 기준 0.4
                 f1_score(y_, hat), accuracy_score(y_, p_ >= 0.5)])      # 원본의 0.5도 함께
cols = ["세트", "인원", "AUC", "Acc(0.4)", "F1(0.4)", "Acc(0.5)"]        # 열 이름
print(pd.DataFrame(rows, columns=cols).round(3).to_string(index=False))  # 원본 셀 56 + 선언한 기준

{'전처리': '숫자 중앙값 + 표준화 · 글자 최빈값 + 원-핫', '피처': '전체 8열', '인코딩': 'OneHot (5-3)', '임곗값': 0.4}


          세트  인원   AUC  Acc(0.4)  F1(0.4)  Acc(0.5)
valid (모의고사) 178 0.857     0.798    0.753     0.798
   test (수능) 179 0.829     0.771    0.717     0.782

**결과** · 예측 정답은 낮음 — AUC 0.857 → 0.829

- `{'전처리': '숫자 중앙값 + 표준화 · 글자 최빈값 + 원-핫', '피처': '전체 8열', '인코딩': 'OneHot (5-3)', '임곗값': 0.4}` — 모든 선택은 valid에서 끝남
- `valid (모의고사) 178 0.857     0.798    0.753     0.798` — 고를 때 쓴 점수
- `test (수능) 179 0.829     0.771    0.717     0.782` — 보고할 점수 · AUC 0.829 · Acc(0.5) 0.782는 원본과 같은 값

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">방법</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">언제 쓰나</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">2회차</td>
<td style="text-align: left;"><strong>모델</strong>을 valid로 고름</td>
<td style="text-align: left;">같은 규칙의 시작</td>
</tr>
<tr>
<td style="text-align: left;">3회차</td>
<td style="text-align: left;"><strong>전처리</strong>까지 Pipeline 안으로 (누수 방지)</td>
<td style="text-align: left;">규칙을 넓힘</td>
</tr>
<tr>
<td style="text-align: left;">4회차</td>
<td style="text-align: left;"><strong>임곗값 · 피처 · 인코딩</strong>까지 valid로</td>
<td style="text-align: left;">새 규칙이 아니라 같은 규칙이 넓어진 것</td>
</tr>
</tbody>
</table>

> ⚠️ **주의**
>
> - test에서는 0.5의 정확도(0.782)가 0.4(0.771)보다 높음 → 그래도 기준을 바꾸지 않음 (바꾸면 test로 고른 것)
> - test가 valid보다 높게 나올 수도 있음 (179명 · 흔들림) → 방향이 아니라 '따로 재야 한다'가 요점
> - 3분할을 안 했다면 임곗값 · 피처를 test에서 고르고 그 점수를 발표 = 답안지를 보고 푼 시험 점수

> ✅ **한 줄 정리** · valid에서 고른 것을 모두 고정한 뒤 test를 한 번 평가해 보고하며, 그 점수가 valid보다 낮은 것은 정상이다.

**딥러닝 코드에서는**

- 하이퍼파라미터 · 기준을 val로 고정 → test 1회 → 보고서 · 논문 표

---
## 그래서 오늘 무엇을 배웠는가?

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">학습 목표</th>
<th style="text-align: left;">할 수 있게 된 것</th>
<th style="text-align: left;">확인 위치</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1. Sigmoid</td>
<td style="text-align: left;">σ(0) = 0.5 · σ(2.3) = 0.909 · 가운데 민감 · 양끝 둔감</td>
<td style="text-align: left;">1-1</td>
</tr>
<tr>
<td style="text-align: left;">2. 확률 · 손실</td>
<td style="text-align: left;"><code>predict</code> = 확률 ≥ 0.5 · 0.99로 틀리면 벌점 6.46배 · 반복 32번</td>
<td style="text-align: left;">2-2 · 2-3</td>
</tr>
<tr>
<td style="text-align: left;">3. 확률의 가치</td>
<td style="text-align: left;">Dummy와 정확도 0.7978 같음 · AUC 0.857은 로지스틱만</td>
<td style="text-align: left;">2-4</td>
</tr>
<tr>
<td style="text-align: left;">4. 임곗값</td>
<td style="text-align: left;">규칙 먼저 → valid F1 최대 0.4 · 정밀도 하락 구간 2곳</td>
<td style="text-align: left;">3-1 ~ 3-3</td>
</tr>
<tr>
<td style="text-align: left;">5. 근거 · test</td>
<td style="text-align: left;">Sex ± 1.3 · Pclass -0.826 · 3등석 여성 0.592 &gt; 1등석 남성 0.436 · test AUC 0.829 · F1 0.717</td>
<td style="text-align: left;">5-2 · 6</td>
</tr>
</tbody>
</table>

### 주제별 한 줄 · 반드시 기억할 것

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">등급</th>
<th style="text-align: left;">주제</th>
<th style="text-align: left;">반드시 기억할 것</th>
<th style="text-align: left;">딥러닝 코드에서</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>A</code></td>
<td style="text-align: left;">Sigmoid</td>
<td style="text-align: left;">z = 0 ⇔ 0.5</td>
<td style="text-align: left;"><code>torch.sigmoid</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>A</code></td>
<td style="text-align: left;">predict_proba</td>
<td style="text-align: left;">확률이 먼저 · predict는 0.5 비교</td>
<td style="text-align: left;"><code>sigmoid(logit)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>A</code></td>
<td style="text-align: left;">교차 엔트로피</td>
<td style="text-align: left;">확신하고 틀리면 벌점 급증</td>
<td style="text-align: left;"><code>BCEWithLogitsLoss</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>A</code></td>
<td style="text-align: left;">기준선 비교</td>
<td style="text-align: left;">정확도 같아도 확률 · AUC</td>
<td style="text-align: left;">AUROC 기록</td>
</tr>
<tr>
<td style="text-align: left;"><code>A</code></td>
<td style="text-align: left;">임곗값</td>
<td style="text-align: left;">규칙 먼저 · valid에서</td>
<td style="text-align: left;">val에서 고른 기준 저장</td>
</tr>
<tr>
<td style="text-align: left;"><code>A</code></td>
<td style="text-align: left;">확률 ≠ 정답 보장</td>
<td style="text-align: left;">강하게 기울었다는 뜻</td>
<td style="text-align: left;">과신 · 보정</td>
</tr>
<tr>
<td style="text-align: left;"><code>A</code></td>
<td style="text-align: left;">3분할 · test 1회</td>
<td style="text-align: left;">고를 것이 늘수록 test를 지킴</td>
<td style="text-align: left;">test loader는 마지막</td>
</tr>
<tr>
<td style="text-align: left;"><code>B</code></td>
<td style="text-align: left;">z = 거리 × ‖w‖</td>
<td style="text-align: left;">순위 같음 · 배율 ‖w‖</td>
<td style="text-align: left;">logit 크기 · weight decay</td>
</tr>
<tr>
<td style="text-align: left;"><code>B</code></td>
<td style="text-align: left;">계수 해석</td>
<td style="text-align: left;">부호는 방향 · 크기 비교는 조심</td>
<td style="text-align: left;">SHAP</td>
</tr>
<tr>
<td style="text-align: left;"><code>B</code></td>
<td style="text-align: left;">상호작용</td>
<td style="text-align: left;">성별마다 나이 효과가 다름</td>
<td style="text-align: left;">은닉층</td>
</tr>
</tbody>
</table>

---
## 기억 확인 문제

위 내용을 **보지 않고** 답한 뒤, ✅ 한 줄 정리와 비교해 ○ (정확) · △ (일부) · × (모름)를 표시. △ · ×인 섹션만 다시 보기.

### 1. 질문에 답하기

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">섹션</th>
<th style="text-align: left;">🎯 질문</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1-1</td>
<td style="text-align: left;">σ(0) · σ(2.3)은?</td>
</tr>
<tr>
<td style="text-align: left;">2-1</td>
<td style="text-align: left;">train · valid · test 인원은?</td>
</tr>
<tr>
<td style="text-align: left;">2-2</td>
<td style="text-align: left;"><code>max_iter=5000</code>인데 실제 반복은?</td>
</tr>
<tr>
<td style="text-align: left;">2-3</td>
<td style="text-align: left;">0.99로 틀린 벌점은 0.51로 틀린 것의 몇 배?</td>
</tr>
<tr>
<td style="text-align: left;">2-4</td>
<td style="text-align: left;">로지스틱이 Dummy보다 나은 점은?</td>
</tr>
<tr>
<td style="text-align: left;">3-1</td>
<td style="text-align: left;">기준을 0.7로 올리면 FN은?</td>
</tr>
<tr>
<td style="text-align: left;">3-2</td>
<td style="text-align: left;">valid F1 최대 기준은?</td>
</tr>
<tr>
<td style="text-align: left;">3-3</td>
<td style="text-align: left;">기준을 올리면 정밀도는 항상 오르나?</td>
</tr>
<tr>
<td style="text-align: left;">4-1</td>
<td style="text-align: left;">확률 0.956인데 사망한 사람의 뜻은?</td>
</tr>
<tr>
<td style="text-align: left;">4-2</td>
<td style="text-align: left;">z와 거리의 관계는?</td>
</tr>
<tr>
<td style="text-align: left;">4-3</td>
<td style="text-align: left;">Pclass만 쓰면 확률이 3개뿐인 이유는?</td>
</tr>
<tr>
<td style="text-align: left;">5-1</td>
<td style="text-align: left;">'아이 먼저'가 뚜렷한 쪽은?</td>
</tr>
<tr>
<td style="text-align: left;">5-2</td>
<td style="text-align: left;">3등석 여성 vs 1등석 남성은?</td>
</tr>
<tr>
<td style="text-align: left;">5-3</td>
<td style="text-align: left;">원-핫을 쓰는 진짜 이유는?</td>
</tr>
</tbody>
</table>

#### 내 답 · 이 셀을 더블클릭해서 작성

**1-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-4** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3-3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**4-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**4-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**4-3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**5-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**5-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**5-3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

### 2. 코드로 답하기

아래 빈 셀에 직접 작성 후 실행.

1. `make_lr_pipeline(["Sex", "Age"])`을 train에 fit해 valid AUC를 소수 셋째 자리로 출력
2. 같은 모델로 기준 0.4의 valid 재현율을 소수 셋째 자리로 출력

In [17]:
# 코드 문제 직접 작성

### 3. 변형 문제 · 조건을 바꿔 다시 풀기

1. 2-3에서 정답이 생존(1)인데 확률 0.01로 예측하면 벌점은? (0.99로 틀린 것과 비교)
2. 3-2 규칙을 '재현율 0.8 이상 중 정밀도 최대'로 바꾸면 기준은?
3. 5-1의 패턴을 넣으려고 `is_child`(12세 이하) 열을 만들면 어떤 열과 곱해야 할까?

In [18]:
# 변형 문제 직접 작성

### 4. 모범 답안 · 1 ~ 3을 다 푼 뒤에 보기

#### 질문 답

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">섹션</th>
<th style="text-align: left;">답</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1-1</td>
<td style="text-align: left;">0.5 · 0.909</td>
</tr>
<tr>
<td style="text-align: left;">2-1</td>
<td style="text-align: left;">534 · 178 · 179 (test를 먼저 봉인)</td>
</tr>
<tr>
<td style="text-align: left;">2-2</td>
<td style="text-align: left;">32번 · max_iter는 상한</td>
</tr>
<tr>
<td style="text-align: left;">2-3</td>
<td style="text-align: left;">6.46배</td>
</tr>
<tr>
<td style="text-align: left;">2-4</td>
<td style="text-align: left;">정확도(0.7978)는 같고, 확률을 줘서 임곗값 · AUC(0.857)가 가능</td>
</tr>
<tr>
<td style="text-align: left;">3-1</td>
<td style="text-align: left;">11 → 33으로 늘어남 (누락 ↑)</td>
</tr>
<tr>
<td style="text-align: left;">3-2</td>
<td style="text-align: left;">0.4</td>
</tr>
<tr>
<td style="text-align: left;">3-3</td>
<td style="text-align: left;">아님 · 양성 예측이 적어지면 흔들림 (0.65 · 0.95)</td>
</tr>
<tr>
<td style="text-align: left;">4-1</td>
<td style="text-align: left;">확률 ≠ 정답 보장 · 강하게 기울었다는 뜻</td>
</tr>
<tr>
<td style="text-align: left;">4-2</td>
<td style="text-align: left;">z = 거리 × ‖w‖ · 순위 같음</td>
</tr>
<tr>
<td style="text-align: left;">4-3</td>
<td style="text-align: left;">Pclass 값이 1 · 2 · 3 세 가지뿐</td>
</tr>
<tr>
<td style="text-align: left;">5-1</td>
<td style="text-align: left;">남성 (아이 0.568 vs 성인 0.173)</td>
</tr>
<tr>
<td style="text-align: left;">5-2</td>
<td style="text-align: left;">3등석 여성 0.592 &gt; 1등석 남성 0.436</td>
</tr>
<tr>
<td style="text-align: left;">5-3</td>
<td style="text-align: left;">성능보다 의미 · 없던 순서를 만들지 않음</td>
</tr>
</tbody>
</table>

#### 코드 문제 답

```python
m_sa = make_lr_pipeline(["Sex", "Age"]).fit(X_train, y_train)
p_sa = m_sa.predict_proba(X_valid)[:, 1]
print(round(roc_auc_score(y_valid, p_sa), 3))  # 0.783
print(round(recall_score(y_valid, p_sa >= 0.4), 3))  # 0.706
```

#### 변형 문제 답

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">문제</th>
<th style="text-align: left;">답</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">1</td>
<td style="text-align: left;">4.6052 — 정답 쪽 확률이 0.01이면 0.99로 틀린 것과 같은 벌점 (대칭)</td>
</tr>
<tr>
<td style="text-align: left;">2</td>
<td style="text-align: left;">0.4 (정밀도 0.7051 · 재현율 0.8088) — 이번엔 F1 최대와 같은 기준</td>
</tr>
<tr>
<td style="text-align: left;">3</td>
<td style="text-align: left;"><code>is_male</code> — 남성 안에서만 아이 효과가 크므로 <code>is_child × is_male</code> 열을 추가</td>
</tr>
</tbody>
</table>

---
## 오답 노트

틀린 문제는 정답을 베끼지 말고 **원인**을 적음. 다음 복습 때 이 표의 "재발 방지 질문"부터 스스로에게 물어보기.

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">틀린 내용</th>
<th style="text-align: left;">틀린 이유</th>
<th style="text-align: left;">올바른 원칙</th>
<th style="text-align: left;">재발 방지 질문</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">(예) predict_proba 0.97 = 97% 맞다고 답함</td>
<td style="text-align: left;">확률을 정답률로 착각</td>
<td style="text-align: left;">확률은 모델의 기울기 · 보정은 따로</td>
<td style="text-align: left;">0.9 이상인데 틀린 사람이 있었나?</td>
</tr>
<tr>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
</tr>
<tr>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
<td style="text-align: left;"></td>
</tr>
</tbody>
</table>

---
## 면접 30초 답변

**Q1. 로지스틱 회귀는 어떻게 확률을 만드나요?**
> - 입력의 가중합으로 점수 z를 계산하고 Sigmoid 1 / (1 + e^-z)로 0과 1 사이의 확률로 바꿉니다.
> - z가 0이면 확률 0.5인 결정 경계이고, 학습은 교차 엔트로피 손실을 줄이도록 가중치를 조정합니다.
> - 확신하고 틀릴수록 벌점이 커서 0.99로 틀리면 0.51로 틀린 것의 6.46배입니다.

**Q2. 정확도가 규칙 기준선과 같은데 로지스틱을 쓸 이유가 있나요?**
> - 타이타닉에서 '남자면 사망' 규칙과 로지스틱의 valid 정확도는 0.7978로 같았습니다.
> - 하지만 규칙은 0과 1만 내서 임곗값을 바꿀 수도, 순위를 매길 수도 없습니다.
> - 로지스틱은 확률을 주어 AUC 0.857로 순위를 평가하고, 업무 정책에 맞게 임곗값을 고를 수 있습니다.

**Q3. 로지스틱 회귀의 계수는 어떻게 해석하나요?**
> - 계수의 부호는 그 특성이 양성 쪽으로 미는지 음성 쪽으로 미는지를 알려 줍니다.
> - 다만 표준화한 숫자 열과 원-핫 열은 단위가 달라 크기만으로 중요도를 단정하지 않습니다.
> - 그래서 조합별 예측 확률처럼 실측으로 확인하며, 3등석 여성이 1등석 남성보다 생존 확률이 높았습니다.

---
## 최종 요약표

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">함수·문법</th>
<th style="text-align: left;">소속</th>
<th style="text-align: left;">역할</th>
<th style="text-align: left;">코드에서 사용한 위치</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>1 / (1 + np.exp(-z))</code></td>
<td style="text-align: left;">numpy</td>
<td style="text-align: left;">Sigmoid</td>
<td style="text-align: left;">1-1 <code>return 1 / (1 + np.exp(-z))</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>ColumnTransformer</code></td>
<td style="text-align: left;">sklearn.compose</td>
<td style="text-align: left;">열마다 다른 전처리</td>
<td style="text-align: left;">2-1 <code>ColumnTransformer([("num", …), ("cat", …)])</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>SimpleImputer</code> · <code>OneHotEncoder</code></td>
<td style="text-align: left;">sklearn</td>
<td style="text-align: left;">결측 · 원-핫</td>
<td style="text-align: left;">2-1 <code>SimpleImputer(strategy="median")</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>LogisticRegression</code></td>
<td style="text-align: left;">sklearn.linear_model</td>
<td style="text-align: left;">로지스틱 회귀</td>
<td style="text-align: left;">2-1 <code>LogisticRegression(max_iter=5000, …)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>predict_proba()[:, 1]</code></td>
<td style="text-align: left;">분류 모델</td>
<td style="text-align: left;">양성 확률</td>
<td style="text-align: left;">2-2 <code>lr.predict_proba(X_valid)[:, 1]</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>n_iter_</code> · <code>solver</code></td>
<td style="text-align: left;">LogisticRegression 속성</td>
<td style="text-align: left;">실제 반복 · 최적화 방법</td>
<td style="text-align: left;">2-2 <code>lr_model.n_iter_[0]</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>-np.log(1 - q)</code></td>
<td style="text-align: left;">numpy</td>
<td style="text-align: left;">정답 0의 교차 엔트로피</td>
<td style="text-align: left;">2-3 <code>loss = -np.log(1 - q)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>roc_auc_score</code></td>
<td style="text-align: left;">sklearn.metrics</td>
<td style="text-align: left;">순위 성능</td>
<td style="text-align: left;">2-4 <code>roc_auc_score(y_valid, y_proba)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>confusion_matrix().ravel()</code></td>
<td style="text-align: left;">sklearn.metrics</td>
<td style="text-align: left;">4칸</td>
<td style="text-align: left;">3-1 <code>confusion_matrix(y_valid, y_hat).ravel()</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>decision_function</code></td>
<td style="text-align: left;">선형 분류 모델</td>
<td style="text-align: left;">점수 z</td>
<td style="text-align: left;">4-2 <code>lr_model.decision_function(Xt)</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>coef_</code> · <code>intercept_</code> · <code>get_feature_names_out</code></td>
<td style="text-align: left;">LogisticRegression · 전처리</td>
<td style="text-align: left;">계수 · 절편 · 열 이름</td>
<td style="text-align: left;">5-2 <code>m_ps.named_steps["model"].coef_[0]</code></td>
</tr>
<tr>
<td style="text-align: left;"><code>OrdinalEncoder(handle_unknown=)</code></td>
<td style="text-align: left;">sklearn.preprocessing</td>
<td style="text-align: left;">정수 라벨</td>
<td style="text-align: left;">5-3 <code>OrdinalEncoder(handle_unknown="use_encoded_value", …)</code></td>
</tr>
</tbody>
</table>

### 할 것 · 하지 말 것

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">✅ 할 것</th>
<th style="text-align: left;">⚠️ 하지 말 것</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">test를 먼저 봉인</td>
<td style="text-align: left;">임곗값 · 피처를 test에서 고르기</td>
</tr>
<tr>
<td style="text-align: left;">기준선과 정확도 · AUC 비교</td>
<td style="text-align: left;">정확도 0.798만 보고 판단</td>
</tr>
<tr>
<td style="text-align: left;">규칙을 표보다 먼저</td>
<td style="text-align: left;">선언한 규칙과 다른 기준으로 최종 평가</td>
</tr>
<tr>
<td style="text-align: left;">확률은 '기울기'로 읽기</td>
<td style="text-align: left;">0.97 = 97% 맞다</td>
</tr>
<tr>
<td style="text-align: left;">계수는 부호 + 실측 확률로</td>
<td style="text-align: left;">계수 크기만으로 중요도 단정</td>
</tr>
</tbody>
</table>

### 헷갈리는 짝 한눈에 보기

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">짝</th>
<th style="text-align: left;">차이</th>
<th style="text-align: left;">결론</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>predict</code> vs <code>predict_proba</code></td>
<td style="text-align: left;">0 / 1 vs 확률</td>
<td style="text-align: left;">기준을 고르려면 proba</td>
</tr>
<tr>
<td style="text-align: left;">z vs 확률</td>
<td style="text-align: left;">-∞ ~ +∞ vs 0 ~ 1</td>
<td style="text-align: left;">확률은 σ(z)</td>
</tr>
<tr>
<td style="text-align: left;">z vs 거리</td>
<td style="text-align: left;">거리 × ‖w‖ vs 거리</td>
<td style="text-align: left;">순위는 같음</td>
</tr>
<tr>
<td style="text-align: left;">정확도 vs AUC</td>
<td style="text-align: left;">기준 하나의 결과 vs 순위 능력</td>
<td style="text-align: left;">둘 다 봄</td>
</tr>
<tr>
<td style="text-align: left;">valid 점수 vs test 점수</td>
<td style="text-align: left;">고를 때 쓴 점수 vs 보고할 점수</td>
<td style="text-align: left;">보고는 test</td>
</tr>
</tbody>
</table>

---
## 다음 복습일

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">언제</th>
<th style="text-align: left;">할 일</th>
<th style="text-align: left;">걸리는 시간</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">내일</td>
<td style="text-align: left;">기억 확인 문제 1을 자료 없이 다시 답하고 오답 노트 갱신</td>
<td style="text-align: left;">15분</td>
</tr>
<tr>
<td style="text-align: left;">3일 뒤</td>
<td style="text-align: left;">변형 문제 3개 · 2-3 벌점 표를 손으로 다시 계산</td>
<td style="text-align: left;">20분</td>
</tr>
<tr>
<td style="text-align: left;">7일 뒤</td>
<td style="text-align: left;">6장을 <code>is_child × is_male</code> 열을 넣어 다시 (선택은 valid에서)</td>
<td style="text-align: left;">30분</td>
</tr>
<tr>
<td style="text-align: left;">14일 뒤</td>
<td style="text-align: left;">오답 노트에 남은 문제만 · 파일별 6(결정 트리) 시작 전 ✅ 줄만 읽기</td>
<td style="text-align: left;">15분</td>
</tr>
</tbody>
</table>

---
## 부록

### 이 노트북을 만든 방법

- 원본 노트북의 셀 60개를 원본 Part 순서대로 1 ~ 5장 · 14개 섹션으로 나눔 (Part 5 · 계수 · 인코딩을 4 · 5장으로)
- 원본 plotly 그림 5개(Sigmoid · 임곗값 곡선 · 확률 분포 2개)는 같은 계산으로 장 그림으로
- 원본 Prediction Card 9개는 각 섹션의 '먼저 예측'으로 · 슬라이드 스크린샷 4장은 💡 · 비교 표의 글로
- 원본 6장이 선언한 규칙(F1 최대) 대신 0.5로 평가한 부분은 6장에서 선언한 기준 0.4로 평가하고 0.5도 함께 표시

### 읽는 법

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">표시</th>
<th style="text-align: left;">의미</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;"><code>A</code> · <code>B</code></td>
<td style="text-align: left;"><strong>A</strong> 딥러닝 코드에 바로 나오는 필수 · <strong>B</strong> 알아두면 되는 것</td>
</tr>
<tr>
<td style="text-align: left;">🎯</td>
<td style="text-align: left;">이번 섹션에서 답해야 할 질문</td>
</tr>
<tr>
<td style="text-align: left;">💡</td>
<td style="text-align: left;">쉬운 설명 · 비유</td>
</tr>
<tr>
<td style="text-align: left;">⚠️</td>
<td style="text-align: left;">자주 하는 실수</td>
</tr>
<tr>
<td style="text-align: left;">✅</td>
<td style="text-align: left;">기억할 결론 = 🎯 질문의 답</td>
</tr>
</tbody>
</table>

- **섹션 앞부분** · 🎯 질문 → 💡 쉬운 설명 → 실행 흐름 → 먼저 예측 → 코드
- **섹션 뒷부분** · 결과(예측 정답 · 출력 줄) → 차이 · 언제 쓰나 → ⚠️ 주의 → ✅ 한 줄 정리
- **코드 주석** · 코드 위는 이 단계에서 하는 일, 코드 옆은 그 줄이 필요한 이유 (줄마다 · 열 맞춤)

### 원본 → 섹션 대응표

<table width="100%" style="width:100%; table-layout:auto">
<thead>
<tr>
<th style="text-align: left;">원본</th>
<th style="text-align: left;">내용</th>
<th style="text-align: left;">이 노트북</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left;">셀 0 ~ 2</td>
<td style="text-align: left;">목표 · 분류란 · 대표 알고리즘</td>
<td style="text-align: left;">0. 원본 목표</td>
</tr>
<tr>
<td style="text-align: left;">셀 3 ~ 5</td>
<td style="text-align: left;">Sigmoid · 그래프 해석</td>
<td style="text-align: left;">1-1 · 1장 그림</td>
</tr>
<tr>
<td style="text-align: left;">셀 6 ~ 9</td>
<td style="text-align: left;">타이타닉 · 3분할 · Pipeline</td>
<td style="text-align: left;">2-1</td>
</tr>
<tr>
<td style="text-align: left;">셀 10 ~ 15</td>
<td style="text-align: left;">로지스틱의 본질 · 학습 · predict_proba · n_iter_</td>
<td style="text-align: left;">2-2</td>
</tr>
<tr>
<td style="text-align: left;">셀 11 · 16 ~ 18</td>
<td style="text-align: left;">교차 엔트로피 · Card 1</td>
<td style="text-align: left;">2-3</td>
</tr>
<tr>
<td style="text-align: left;">셀 19 ~ 22</td>
<td style="text-align: left;">기준선 · Card 2 · 충격</td>
<td style="text-align: left;">2-4</td>
</tr>
<tr>
<td style="text-align: left;">셀 23 ~ 25</td>
<td style="text-align: left;">기준 0.3 · 0.5 · 0.7</td>
<td style="text-align: left;">3-1</td>
</tr>
<tr>
<td style="text-align: left;">셀 26 ~ 29</td>
<td style="text-align: left;">선택 규칙 · 표 · Card 3</td>
<td style="text-align: left;">3-2 · 3장 그림</td>
</tr>
<tr>
<td style="text-align: left;">셀 29 ~ 31</td>
<td style="text-align: left;">Card 4 · 줄다리기는 경향</td>
<td style="text-align: left;">3-3</td>
</tr>
<tr>
<td style="text-align: left;">셀 32 ~ 37</td>
<td style="text-align: left;">확률 분포 · Card 5 · 확신하고 틀림</td>
<td style="text-align: left;">4-1</td>
</tr>
<tr>
<td style="text-align: left;">셀 41 ~ 43</td>
<td style="text-align: left;">z = 거리 × ‖w‖</td>
<td style="text-align: left;">4-2</td>
</tr>
<tr>
<td style="text-align: left;">셀 38 ~ 40 · 44 · 51 · 52</td>
<td style="text-align: left;">Card 6 · 피처를 늘리면 · Sex의 힘</td>
<td style="text-align: left;">4-3 · 4장 그림</td>
</tr>
<tr>
<td style="text-align: left;">셀 45 ~ 47</td>
<td style="text-align: left;">Card 7 · 아이</td>
<td style="text-align: left;">5-1</td>
</tr>
<tr>
<td style="text-align: left;">셀 48 ~ 50</td>
<td style="text-align: left;">Card 8 · 계수</td>
<td style="text-align: left;">5-2 · 5장 그림</td>
</tr>
<tr>
<td style="text-align: left;">셀 52 ~ 54</td>
<td style="text-align: left;">인코딩 비교</td>
<td style="text-align: left;">5-3</td>
</tr>
<tr>
<td style="text-align: left;">셀 55 ~ 57</td>
<td style="text-align: left;">수능 · Card 9 · 결과 해석</td>
<td style="text-align: left;">6. 종합 실습</td>
</tr>
<tr>
<td style="text-align: left;">셀 58 · 59</td>
<td style="text-align: left;">오늘의 정리 · 다음 4-2 KNN</td>
<td style="text-align: left;">그래서 오늘 · 다음 복습일</td>
</tr>
</tbody>
</table>

### 전체 코드

1 ~ 6장 코드를 장별 셀로 모음. **위에서부터 이 셀들만 실행해도 전체가 동작**하며, 각 블록 머리 주석의 번호가 섹션.

In [19]:
# ---------- 1-1 ----------
def sigmoid(z):                  # 점수 → 확률
    return 1 / (1 + np.exp(-z))  # 원본 수식


for z in [-10, -2, 0, 2, 2.3, 10]:                           # 원본 표의 값 + 예시 2.3
    print(f"z = {z:>5} → σ(z) = {sigmoid(z):.3f}")           # 0 ~ 1
print("가운데 z 0 → 1:", round(sigmoid(1) - sigmoid(0), 3))  # 민감
print("끝쪽 z 5 → 6:", round(sigmoid(6) - sigmoid(5), 3))    # 둔감

z =   -10 → σ(z) = 0.000
z =    -2 → σ(z) = 0.119
z =     0 → σ(z) = 0.500
z =     2 → σ(z) = 0.881
z =   2.3 → σ(z) = 0.909
z =    10 → σ(z) = 1.000
가운데 z 0 → 1: 0.231
끝쪽 z 5 → 6: 0.004


In [20]:
# ---------- 2-1 ----------
y_all = titanic_df["Survived"]                                          # 정답
X_all = titanic_df.drop("Survived", axis=1).copy()                      # 문제
X_all["Cabin1"] = X_all["Cabin"].fillna("N").astype(str).str[:1]        # 선실 첫 글자 (없으면 N)
X_all = X_all.drop(["PassengerId", "Name", "Ticket", "Cabin"], axis=1)  # 쓰지 않는 열
print("데이터 크기:", X_all.shape, "· 생존 비율:", round(y_all.mean(), 4))  # 8열
X_dev, X_final_test, y_dev, y_final_test = train_test_split(            # 1단계: 수능 봉인
    X_all, y_all, test_size=0.2, stratify=y_all, random_state=42)
X_train, X_valid, y_train, y_valid = train_test_split(                  # 2단계: 교과서 · 모의고사
    X_dev, y_dev, test_size=0.25, stratify=y_dev, random_state=42)
print(f"train {len(X_train)} · valid {len(X_valid)} · test {len(X_final_test)}")  # 6 : 2 : 2
NUM = ["Pclass", "Age", "SibSp", "Parch", "Fare"]                       # 숫자 열
CAT = ["Sex", "Embarked", "Cabin1"]                                     # 글자 열


def make_lr_pipeline(feats):  # 고른 열만으로 Pipeline (4-3 · 5장에서도 씀)
    num = [f for f in feats if f in NUM]                                      # 고른 숫자 열
    cat = [f for f in feats if f in CAT]                                      # 고른 글자 열
    num_tf = Pipeline([("imputer", SimpleImputer(strategy="median")),         # 중앙값 + 표준화
                       ("scaler", StandardScaler())])
    cat_tf = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),  # 최빈값 + 원-핫
                       ("onehot", OneHotEncoder(handle_unknown="ignore"))])
    pre = ColumnTransformer([("num", num_tf, num), ("cat", cat_tf, cat)])     # 열마다 따로
    model = LogisticRegression(max_iter=5000, random_state=42)                # 원본 설정
    return Pipeline([("preprocess", pre), ("model", model)])                  # 전처리 + 모델


ALL = NUM + CAT             # 전체 8열
lr = make_lr_pipeline(ALL)  # 오늘의 기본 모델


# ---------- 2-2 ----------
lr.fit(X_train, y_train)                                            # train에만 fit (전처리 포함)
y_proba = lr.predict_proba(X_valid)[:, 1]                           # 생존 확률 (열 1)
print("예측 확률 (생존) 앞 10명:", y_proba[:10].round(4).tolist())  # 원본 셀 12
lr_model = lr.named_steps["model"]                                  # Pipeline 안의 로지스틱
print(f"solver          : {lr_model.solver}")                       # 최적화 방법
print(f"max_iter (설정) : {lr_model.max_iter}")                     # 상한
print(f"n_iter_  (실제) : {lr_model.n_iter_[0]}")                   # 실제로 돈 횟수
same = (lr.predict(X_valid) == (y_proba >= 0.5)).all()              # predict = 확률 ≥ 0.5?
print("predict == (확률 ≥ 0.5):", same)                             # 내부에서 자동 비교


# ---------- 2-3 ----------
for q in [0.51, 0.7, 0.9, 0.99]:                               # 정답 0인데 낸 생존 확률
    loss = -np.log(1 - q)                                      # 정답(0) 쪽 확률의 -log
    print(f"정답=사망(0)인데 생존 확률 {q:.2f} 로 예측 → 벌점 {loss:.4f}")  # 원본 형식
ratio = -np.log(1 - 0.99) / -np.log(1 - 0.51)                  # 두 벌점의 비
print(f"0.99로 틀린 벌점 / 0.51로 틀린 벌점 = {ratio:.2f}배")  # 확신의 대가


# ---------- 2-4 ----------
def dummy_rule(X):                                        # 3-2 Part 1의 규칙
    return np.where(X["Sex"].to_numpy() == "male", 0, 1)  # 남자면 사망

데이터 크기: (891, 8) · 생존 비율: 0.3838
train 534 · valid 178 · test 179
예측 확률 (생존) 앞 10명: [0.1093, 0.1176, 0.74, 0.1017, 0.084, 0.1017, 0.084, 0.9173, 0.0842, 0.557]
solver          : lbfgs
max_iter (설정) : 5000
n_iter_  (실제) : 32
predict == (확률 ≥ 0.5): True
정답=사망(0)인데 생존 확률 0.51 로 예측 → 벌점 0.7133
정답=사망(0)인데 생존 확률 0.70 로 예측 → 벌점 1.2040
정답=사망(0)인데 생존 확률 0.90 로 예측 → 벌점 2.3026
정답=사망(0)인데 생존 확률 0.99 로 예측 → 벌점 4.6052
0.99로 틀린 벌점 / 0.51로 틀린 벌점 = 6.46배


In [21]:
acc_lr = accuracy_score(y_valid, (y_proba >= 0.5).astype(int))           # 기본 기준
acc_major = accuracy_score(y_valid, np.zeros(len(y_valid)))              # ① 전부 사망
acc_rule = accuracy_score(y_valid, dummy_rule(X_valid))                  # ② 규칙
auc_lr = round(roc_auc_score(y_valid, y_proba), 4)                       # ③ 순위 성능
rows = [["① 전부 사망 (다수 클래스)", acc_major, "-"],                   # 원본 셀 21의 표
        ["② 3-2 Dummy 규칙 (남자=사망)", acc_rule, "확률 없음"],
        ["③ 로지스틱 회귀 (피처 8개)", acc_lr, auc_lr]]
cols = ["모델", "valid Accuracy", "AUC"]                                 # 열 이름
print(pd.DataFrame(rows, columns=cols).round(4).to_string(index=False))  # 비교표

                    모델  valid Accuracy    AUC
      ① 전부 사망 (다수 클래스)          0.6180      -
② 3-2 Dummy 규칙 (남자=사망)          0.7978  확률 없음
     ③ 로지스틱 회귀 (피처 8개)          0.7978  0.857


In [22]:
# ---------- 3-1 ----------
for thr in [0.3, 0.5, 0.7]:                                               # 원본 셀 24의 세 기준
    y_hat = (y_proba >= thr).astype(int)                                  # 확률 → 0 / 1
    tn, fp, fn, tp = confusion_matrix(y_valid, y_hat).ravel()             # 4칸
    p, r = precision_score(y_valid, y_hat), recall_score(y_valid, y_hat)  # 두 지표
    print(f"thr={thr} · P {p:.4f} · R {r:.4f}")                           # 두 지표
    print(f"  FP(오탐) {fp} · FN(누락) {fn} · TP {tp} · TN {tn}")         # 4칸


# ---------- 3-2 ----------
thr_list = np.round(np.arange(0.1, 1.0, 0.1), 1)                                 # 후보 9개
rows = []                                                                        # 기준마다 지표
for thr in thr_list:                                                             # 원본 셀 27
    y_hat = (y_proba >= thr).astype(int)                                         # 0 / 1
    rows.append([thr, precision_score(y_valid, y_hat, zero_division=0),          # 세 지표
                 recall_score(y_valid, y_hat), f1_score(y_valid, y_hat)])
thr_df = pd.DataFrame(rows, columns=["Threshold", "Precision", "Recall", "F1"])  # 표
print(thr_df.round(4).to_string(index=False))                                    # 원본 표
chosen_thr = float(thr_df.loc[thr_df["F1"].idxmax(), "Threshold"])               # 선언한 규칙
print("valid F1 최대 기준:", chosen_thr)                                         # 6장에서 그대로 씀


# ---------- 3-3 ----------
fine = np.round(np.arange(0.05, 0.96, 0.05), 2)         # 촘촘한 기준 19개
prev, drops = None, []                                  # 앞 정밀도 · 내려간 곳
for t in fine:                                          # 기준을 올리며
    yh = (y_proba >= t).astype(int)                     # 0 / 1
    pr = precision_score(y_valid, yh, zero_division=0)  # 정밀도
    if prev is not None and pr < prev - 1e-9:           # 앞보다 낮아졌으면
        drops.append(f"thr={t} · {prev:.4f} → {pr:.4f} · 양성 예측 {int(yh.sum())}명")  # 기록
    prev = pr                                           # 다음 비교용
print(f"정밀도가 내려간 구간: {len(drops)}곳")          # 원본 셀 30
for d in drops:                                         # 하나씩
    print(" ", d)                                       # 위치 · 남은 양성 수

thr=0.3 · P 0.6706 · R 0.8382
  FP(오탐) 28 · FN(누락) 11 · TP 57 · TN 82
thr=0.5 · P 0.7286 · R 0.7500
  FP(오탐) 19 · FN(누락) 17 · TP 51 · TN 91


thr=0.7 · P 0.8140 · R 0.5147
  FP(오탐) 8 · FN(누락) 33 · TP 35 · TN 102


 Threshold  Precision  Recall     F1
       0.1     0.4539  0.9412 0.6124
       0.2     0.6413  0.8676 0.7375
       0.3     0.6706  0.8382 0.7451
       0.4     0.7051  0.8088 0.7534
       0.5     0.7286  0.7500 0.7391
       0.6     0.8036  0.6618 0.7258
       0.7     0.8140  0.5147 0.6306
       0.8     0.8889  0.3529 0.5053
       0.9     0.9474  0.2647 0.4138
valid F1 최대 기준: 0.4


정밀도가 내려간 구간: 2곳
  thr=0.65 · 0.8036 → 0.7708 · 양성 예측 48명
  thr=0.95 · 0.9474 → 0.9167 · 양성 예측 12명


In [23]:
# ---------- 4-1 ----------
chk = pd.DataFrame({"예측 확률": y_proba.round(3), "실제": y_valid.to_numpy()})  # 원본 셀 36
chk["예측 라벨"] = (chk["예측 확률"] >= 0.5).astype(int)                    # 기본 기준
wrong = chk[chk["예측 라벨"] != chk["실제"]]                                # 틀린 사람
extreme = wrong[(wrong["예측 확률"] >= 0.9) | (wrong["예측 확률"] <= 0.1)]  # 확신하고 틀림
print(f"전체 valid {len(chk)}명 중 틀린 예측: {len(wrong)}명")              # 원본 형식
print(f"그중 모델이 '확신'했는데 틀린 경우: {len(extreme)}명")              # 극단
print(extreme.sort_values("예측 확률").to_string())                         # 누구인지
bins = pd.cut(chk["예측 확률"], [0, 0.3, 0.7, 1.0], include_lowest=True)    # 낮음 · 중간 · 높음
by_bin = chk.groupby(bins, observed=True)["실제"].agg(["size", "mean"])     # 구간별 인원 · 생존률
print(by_bin.round(3).to_string())  # 확률이 높아도 전부 생존은 아님


# ---------- 4-2 ----------
Xt = lr.named_steps["preprocess"].transform(X_valid)                  # 전처리된 입력
Xt = Xt.toarray() if hasattr(Xt, "toarray") else Xt                   # 희소 → 배열
z = lr_model.decision_function(Xt)                                    # 점수 z
w_norm = np.linalg.norm(lr_model.coef_[0])                            # 계수 벡터의 길이
dist = z / w_norm                                                     # 선까지의 실제 거리
print(f"‖w‖ = {w_norm:.4f}")                                          # 배율
print(f"z    범위 : {z.min():.3f} ~ {z.max():.3f}")                   # 점수
print(f"거리  범위 : {dist.min():.3f} ~ {dist.max():.3f}")            # 거리
print("σ(z) = predict_proba:", np.allclose(sigmoid(z), y_proba))      # 확률은 z로 만듦
same_auc = roc_auc_score(y_valid, z) == roc_auc_score(y_valid, dist)  # 순위 지표 비교
print("AUC(z) = AUC(거리):", same_auc)                                # 비례하면 순위가 같음


# ---------- 4-3 ----------
configs = [("Pclass만", ["Pclass"]), ("Sex만", ["Sex"]),                      # 원본 셀 51의 조합
           ("Pclass + Sex", ["Pclass", "Sex"]), ("전체 피처", ALL)]
rows, probs = [], {}                                                          # 표 · 확률 보관
for name, feats in configs:                                                   # 조합마다
    m = make_lr_pipeline(feats).fit(X_train, y_train)                         # 같은 절차
    p = m.predict_proba(X_valid)[:, 1]                                        # valid 확률
    probs[name] = p                                                           # 몇 종류인지 볼 때 씀
    acc_ = accuracy_score(y_valid, p >= 0.5)                                  # 기본 기준 정확도
    rows.append([name, len(feats), roc_auc_score(y_valid, p), acc_])          # 지표
cols = ["모델", "원본 컬럼 수", "AUC", "Accuracy"]                            # 원본 열 이름
print(pd.DataFrame(rows, columns=cols).round(3).to_string(index=False))       # 원본 표
print("Pclass만 확률 값:", sorted(set(probs["Pclass만"].round(3).tolist())))  # 몇 종류?

전체 valid 178명 중 틀린 예측: 36명
그중 모델이 '확신'했는데 틀린 경우: 5명
     예측 확률  실제  예측 라벨
108  0.075   1      0
54   0.089   1      0
105  0.100   1      0
162  0.100   1      0
43   0.956   0      1
               size   mean
예측 확률                     
(-0.001, 0.3]    93  0.118
(0.3, 0.7]       42  0.524
(0.7, 1.0]       43  0.814
‖w‖ = 2.5839
z    범위 : -3.304 ~ 4.086
거리  범위 : -1.279 ~ 1.581
σ(z) = predict_proba: True
AUC(z) = AUC(거리): True


          모델  원본 컬럼 수   AUC  Accuracy
     Pclass만        1 0.702     0.669
        Sex만        1 0.780     0.798
Pclass + Sex        2 0.838     0.798
       전체 피처        8 0.857     0.798
Pclass만 확률 값: [0.248, 0.437, 0.647]


In [24]:
from sklearn.preprocessing import OrdinalEncoder  # 정수 라벨


# ---------- 5-1 ----------
age_df = titanic_df.dropna(subset=["Age"]).copy()                            # 나이를 아는 714명
age_df["연령대"] = np.where(age_df["Age"] <= 12, "아이(0~12)", "성인(13+)")  # 두 그룹
grp = age_df.groupby(["Sex", "연령대"])["Survived"]                          # 성별 × 연령대
tbl = grp.agg(인원="size", 생존률="mean")                                    # 인원 · 생존률
print(tbl.round(3).to_string())                                              # 원본 셀 46
print("나이 아는 인원:", len(age_df), "· 결측:", titanic_df["Age"].isna().sum())  # 대상 확인


# ---------- 5-2 ----------
m_ps = make_lr_pipeline(["Pclass", "Sex"]).fit(X_train, y_train)     # 피처 2개
names = m_ps.named_steps["preprocess"].get_feature_names_out()       # 전처리 후 열 이름
coefs = m_ps.named_steps["model"].coef_[0]                           # 계수 w
coef_df = pd.DataFrame({"피처": names, "계수 w": coefs.round(3)})    # 열 이름 + 계수
print(coef_df.to_string(index=False))                                # 부호 확인
print(f"절편 b: {m_ps.named_steps['model'].intercept_[0]:.3f}")      # 기준점
combo = X_valid[["Pclass", "Sex"]].drop_duplicates()                 # 6조합
combo["예측 확률(생존)"] = m_ps.predict_proba(combo)[:, 1].round(3)  # 조합별 확률
print(combo.sort_values(["Sex", "Pclass"]).to_string(index=False))   # 원본 셀 49


# ---------- 5-3 ----------


def make_pipeline_with(enc):                                               # 인코더만 바꾼 Pipeline
    num_tf = Pipeline([("imputer", SimpleImputer(strategy="median")),      # 숫자는 같음
                       ("scaler", StandardScaler())])
    cat_tf = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),  # 인코더만 다름
                       ("enc", enc)])
    pre = ColumnTransformer([("num", num_tf, NUM), ("cat", cat_tf, CAT)])  # 같은 열 구성
    model = LogisticRegression(max_iter=5000, random_state=42)             # 같은 모델
    return Pipeline([("preprocess", pre), ("model", model)])               # 전처리 + 모델


ord_enc = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)   # 처음 본 값은 -1
encs = [("정수 라벨 (예전 방식)", ord_enc),                                      # 비교할 두 인코더
        ("OneHot (지금 방식)", OneHotEncoder(handle_unknown="ignore"))]
for label, enc in encs:                                                          # 같은 절차
    mm = make_pipeline_with(enc).fit(X_train, y_train)                           # train에 fit
    pp = mm.predict_proba(X_valid)[:, 1]                                         # valid 확률
    dim = mm.named_steps["preprocess"].transform(X_train).shape[1]               # 모델이 보는 차원
    auc_, acc_ = roc_auc_score(y_valid, pp), accuracy_score(y_valid, pp >= 0.5)  # 두 지표
    print(f"{label} · 차원 {dim} · AUC {auc_:.4f} · Acc {acc_:.4f}")             # 원본 셀 53의 표

                  인원    생존률
Sex    연령대                 
female 성인(13+)   229  0.777
       아이(0~12)   32  0.594
male   성인(13+)   416  0.173
       아이(0~12)   37  0.568
나이 아는 인원: 714 · 결측: 177
             피처   계수 w
    num__Pclass -0.826
cat__Sex_female  1.307
  cat__Sex_male -1.306
절편 b: -0.248
 Pclass    Sex  예측 확률(생존)
      1 female      0.913
      2 female      0.796
      3 female      0.592
      1   male      0.436
      2   male      0.223
      3   male      0.096
정수 라벨 (예전 방식) · 차원 8 · AUC 0.8566 · Acc 0.7809


OneHot (지금 방식) · 차원 19 · AUC 0.8570 · Acc 0.7978


In [25]:
# ---------- 6. 종합 실습 ----------
# 1 · 2단계: valid에서 고른 것을 고정하고 train에 fit
choices = {"전처리": "숫자 중앙값 + 표준화 · 글자 최빈값 + 원-핫",  # 전부 valid로 고름
           "피처": "전체 8열", "인코딩": "OneHot (5-3)", "임곗값": chosen_thr}
print(choices)                                                      # 고정한 선택
final_model = make_lr_pipeline(ALL).fit(X_train, y_train)           # 같은 설정 · 같은 train


# 3 · 4단계: valid 점수(참고)와 test 1회
rows = []                                                                # 세트별 점수
sets = [("valid (모의고사)", X_valid, y_valid),                          # 참고용
        ("test (수능)", X_final_test, y_final_test)]                     # 처음이자 마지막
for name, X_, y_ in sets:                                                # 같은 지표로
    p_ = final_model.predict_proba(X_)[:, 1]                             # 확률
    hat = p_ >= chosen_thr                                               # 선언한 기준 0.4
    rows.append([name, len(y_), roc_auc_score(y_, p_), accuracy_score(y_, hat),  # 기준 0.4
                 f1_score(y_, hat), accuracy_score(y_, p_ >= 0.5)])      # 원본의 0.5도 함께
cols = ["세트", "인원", "AUC", "Acc(0.4)", "F1(0.4)", "Acc(0.5)"]        # 열 이름
print(pd.DataFrame(rows, columns=cols).round(3).to_string(index=False))  # 원본 셀 56 + 선언한 기준

{'전처리': '숫자 중앙값 + 표준화 · 글자 최빈값 + 원-핫', '피처': '전체 8열', '인코딩': 'OneHot (5-3)', '임곗값': 0.4}


          세트  인원   AUC  Acc(0.4)  F1(0.4)  Acc(0.5)
valid (모의고사) 178 0.857     0.798    0.753     0.798
   test (수능) 179 0.829     0.771    0.717     0.782
